# Sky Coefficient Prediction And Reconstruction

This notebook starts the new problem setup:
- Inputs: simultaneous SkyE and SkyW (or near/far) observations
- Intermediate target: science-location decomposition coefficients
- Final target: reconstructed science-location sky spectrum

It reuses robust FITS/decomposition I/O and reconstruction utilities from the generic model notebook, then builds a triplet dataset for coefficient transfer and spectrum reconstruction experiments.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from astropy.io import fits
from astropy.table import Table
from IPython.display import HTML, display

from sky_decomp.fit import reconstruct_component_spectra

# Reused constants
FACTOR = 1e14
PALACE_DIR = '../'

In [ ]:
# Reused decomposition/context loading helpers
def _as_array(x):
    arr = np.asarray(x)
    if arr.dtype.kind in ('U', 'S', 'O'):
        return None
    return arr.astype(np.float32)


def _coerce_coef_hdu_to_table(coef_hdu):
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)

    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D COEF image, got shape={arr.shape}')

    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f'COEF{i:04d}'
        names.append(str(coef_hdu.header.get(key, f'coef_{i:04d}')))
    return Table({name: arr[:, i] for i, name in enumerate(names)})


def _select_context_from_labels(meta, meta_upper, labels, base_name):
    e_key = f'SKYE_{base_name.upper()}'
    w_key = f'SKYW_{base_name.upper()}'
    if e_key not in meta_upper or w_key not in meta_upper:
        return None

    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f'Labeled context columns for {base_name} are non-numeric.')

    is_e = labels == 'SKYE'
    is_w = labels == 'SKYW'
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f'Unexpected label values: {bad}')

    return np.where(is_e, arr_e, arr_w).astype(np.float32)


def _table_to_float32_matrix(tbl, value_name):
    names = list(tbl.colnames)
    cols = []
    numeric_names = []
    for name in names:
        arr = _as_array(tbl[name])
        if arr is not None:
            cols.append(arr)
            numeric_names.append(name)

    if len(cols) == 0:
        raise ValueError(f'No numeric {value_name} columns found.')

    return np.column_stack(cols).astype(np.float32), numeric_names


def _extract_obstime_mjd(meta, meta_upper):
    from astropy.time import Time

    candidates = ('OBSTIME', 'MJD', 'MJD_OBS', 'MJD-OBS', 'DATE_OBS', 'DATE-OBS')
    for key in candidates:
        if key not in meta_upper:
            continue

        raw = np.asarray(meta[meta_upper[key]])
        if raw.dtype.kind in ('i', 'u', 'f'):
            mjd = raw.astype(np.float64)
            if key.startswith('MJD'):
                return mjd
            # Numeric OBSTIME may already be MJD in some products.
            return mjd

        s = np.asarray(raw).astype(str)
        try:
            return Time(s, format='isot', scale='utc').mjd.astype(np.float64)
        except Exception:
            return Time(s).mjd.astype(np.float64)

    raise KeyError('Could not find an OBSTIME/MJD/DATE-OBS-like META column for time features')


def _build_obstime_feature(meta, meta_upper, feature_name):
    mjd = _extract_obstime_mjd(meta, meta_upper)
    two_pi = 2.0 * np.pi

    if feature_name == 'obstime_mjd':
        return mjd.astype(np.float32)

    if feature_name == 'obstime_mjd_z':
        med = np.nanmedian(mjd)
        q25 = np.nanpercentile(mjd, 25.0)
        q75 = np.nanpercentile(mjd, 75.0)
        iqr = q75 - q25
        if not np.isfinite(iqr) or iqr < 1e-8:
            iqr = 1.0
        return ((mjd - med) / iqr).astype(np.float32)

    if feature_name == 'obstime_year_sin':
        return np.sin(two_pi * mjd / 365.2422).astype(np.float32)

    if feature_name == 'obstime_year_cos':
        return np.cos(two_pi * mjd / 365.2422).astype(np.float32)

    frac_day = mjd - np.floor(mjd)
    if feature_name == 'obstime_day_sin':
        return np.sin(two_pi * frac_day).astype(np.float32)

    if feature_name == 'obstime_day_cos':
        return np.cos(two_pi * frac_day).astype(np.float32)

    raise KeyError(f'Unsupported obstime feature name: {feature_name}')


def _build_context_matrix(meta, context_columns, kind):
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None
    obstime_feature_names = {
        'obstime_mjd',
        'obstime_mjd_z',
        'obstime_year_sin',
        'obstime_year_cos',
        'obstime_day_sin',
        'obstime_day_cos',
    }
    obstime_cache = {}

    if kind in ('sky1', 'sky2'):
        label_col = 'SKY_NEAR_LABEL' if kind == 'sky1' else 'SKY_FAR_LABEL'
        if label_col not in meta_upper:
            raise KeyError(f'Missing required META label column: {label_col}')
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []

    for cname in context_columns:
        if cname in obstime_feature_names:
            if cname not in obstime_cache:
                obstime_cache[cname] = _build_obstime_feature(meta, meta_upper, cname)
            ctx_names.append(cname)
            ctx_cols.append(obstime_cache[cname])
            continue

        key = cname.upper()

        # Science rows often use SCI_<name> naming; check this first for sci mode.
        if kind == 'sci':
            sci_key = f'SCI_{key}'
            if sci_key in meta_upper:
                arr = _as_array(meta[meta_upper[sci_key]])
                if arr is None:
                    raise ValueError(f'Context column {sci_key} is non-numeric.')
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue

        if key in meta_upper:
            arr = _as_array(meta[meta_upper[key]])
            if arr is None:
                raise ValueError(f'Context column {cname} is non-numeric.')
            ctx_names.append(cname)
            ctx_cols.append(arr)
            continue

        if labels is not None:
            arr = _select_context_from_labels(meta, meta_upper, labels, cname)
            if arr is not None:
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue

        missing_cols.append(cname)

    if missing_cols:
        raise KeyError(f'Missing requested context columns: {missing_cols}')
    if len(ctx_cols) == 0:
        raise ValueError('No usable context columns were assembled.')

    return np.column_stack(ctx_cols).astype(np.float32), ctx_names


def _find_chi2_column(meta_tbl):
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ('REDUCED_CHI2', 'CHI2_REDUCED', 'CHI2', 'RCHI2'):
        if cand in names:
            return names[cand]
    raise KeyError('No chi2-like column found in decomposition META table')


def read_decomp_dataset(decomp_fits_path, input_fits_path, context_columns, decomp_kind='sky1', return_chi2=False):
    if context_columns is None or len(context_columns) == 0:
        raise ValueError('context_columns must be a non-empty list.')

    kind = decomp_kind.lower()
    if kind not in ('sky1', 'sky2', 'sci'):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_mat, coef_names = _table_to_float32_matrix(coef_tbl, 'coefficient')

        meta = Table(hdul_in['META'].data)
        ctx_mat, ctx_names = _build_context_matrix(meta, context_columns, kind)

        if coef_mat.shape[0] != ctx_mat.shape[0]:
            raise ValueError(
                f'Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows'
            )

        good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
        coef_mat = coef_mat[good]
        ctx_mat = ctx_mat[good]

        if not return_chi2:
            return coef_mat, ctx_mat, coef_names, ctx_names

        dec_meta = Table(hdul_dec['META'].data)
        chi2_col = _find_chi2_column(dec_meta)
        chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
        chi2_used = chi2_full[good]

        if chi2_used.shape[0] != coef_mat.shape[0]:
            raise ValueError(
                f'Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})'
            )

        return coef_mat, ctx_mat, coef_names, ctx_names, chi2_used

In [ ]:
# New helper: build simultaneous triplets (near, far -> sci) from decomposition files
def build_triplet_coef_dataset(
    input_fits_path,
    sky_near_decomp_fits_path,
    sky_far_decomp_fits_path,
    sci_decomp_fits_path,
    context_columns,
    return_chi2=False,
):
    """Build aligned triplet arrays for coefficient-transfer experiments.

    Returns
    -------
    dict with keys:
      coef_near, coef_far, coef_sci
      ctx_near, ctx_far, ctx_sci
      coef_names, ctx_names, n_rows
      optional chi2_near, chi2_far, chi2_sci
    """
    near = read_decomp_dataset(
        decomp_fits_path=sky_near_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sky1',
        return_chi2=return_chi2,
    )
    far = read_decomp_dataset(
        decomp_fits_path=sky_far_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sky2',
        return_chi2=return_chi2,
    )
    sci = read_decomp_dataset(
        decomp_fits_path=sci_decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind='sci',
        return_chi2=return_chi2,
    )

    if return_chi2:
        coef_near, ctx_near, coef_names_n, ctx_names_n, chi2_near = near
        coef_far, ctx_far, coef_names_f, ctx_names_f, chi2_far = far
        coef_sci, ctx_sci, coef_names_s, ctx_names_s, chi2_sci = sci
    else:
        coef_near, ctx_near, coef_names_n, ctx_names_n = near
        coef_far, ctx_far, coef_names_f, ctx_names_f = far
        coef_sci, ctx_sci, coef_names_s, ctx_names_s = sci

    if coef_names_n != coef_names_f or coef_names_n != coef_names_s:
        raise ValueError('Coefficient name mismatch across near/far/sci decomposition products.')
    if ctx_names_n != ctx_names_f or ctx_names_n != ctx_names_s:
        raise ValueError('Context name mismatch across near/far/sci products.')

    n = min(coef_near.shape[0], coef_far.shape[0], coef_sci.shape[0])
    if n == 0:
        raise ValueError('No aligned rows available after filtering.')

    out = {
        'coef_near': coef_near[:n],
        'coef_far': coef_far[:n],
        'coef_sci': coef_sci[:n],
        'ctx_near': ctx_near[:n],
        'ctx_far': ctx_far[:n],
        'ctx_sci': ctx_sci[:n],
        'coef_names': coef_names_n,
        'ctx_names': ctx_names_n,
        'n_rows': n,
    }

    if return_chi2:
        out['chi2_near'] = chi2_near[:n]
        out['chi2_far'] = chi2_far[:n]
        out['chi2_sci'] = chi2_sci[:n]

    print(
        f"Triplet dataset built: n_rows={n}, n_coef={out['coef_near'].shape[1]}, n_ctx={out['ctx_near'].shape[1]}"
    )
    return out

In [ ]:
# Reused reconstruction/prediction helpers for quick visual checks
def _meta_row_to_dict_upper(meta_row):
    names = list(meta_row.colnames) if hasattr(meta_row, 'colnames') else list(meta_row.dtype.names)
    return {str(k).upper(): k for k in names}


def _safe_float(x):
    arr = np.asarray(x)
    if arr.size == 0:
        raise ValueError('Empty value cannot be converted to float')
    if arr.shape != ():
        arr = arr.ravel()[0]
    return float(arr)


def _read_ext_row(hdul, extname, row_index):
    if extname not in [h.name for h in hdul]:
        raise KeyError(f'Missing required extension: {extname}')
    arr = np.asarray(hdul[extname].data, dtype=float)
    if arr.ndim == 1:
        return arr
    if arr.ndim >= 2:
        if row_index < 0 or row_index >= arr.shape[0]:
            raise IndexError(f'row_index {row_index} out of range [0, {arr.shape[0]-1}] for {extname}')
        return np.asarray(arr[row_index], dtype=float)
    raise ValueError(f'Unsupported ndim={arr.ndim} for extension {extname}')


def _display_scrollable_table(tbl):
    try:
        if hasattr(tbl, 'to_pandas'):
            df = tbl.to_pandas()
        else:
            df = pd.DataFrame(tbl)
    except Exception:
        print(tbl)
        return

    html = df.to_html(index=False)
    display(
        HTML(
            "<div style='max-width:100%; overflow-x:auto; border:1px solid #ddd; padding:6px;'>"
            + html
            + "</div>"
        )
    )


def _context_from_meta_row(meta_row, ctx_names_local, mode):
    umap = _meta_row_to_dict_upper(meta_row)
    mode = str(mode).lower()
    if mode not in ('near', 'far', 'sci'):
        raise ValueError(f'Unsupported mode: {mode}')

    label = None
    if mode == 'near' and 'SKY_NEAR_LABEL' in umap:
        label = str(meta_row[umap['SKY_NEAR_LABEL']]).strip().upper()
    elif mode == 'far' and 'SKY_FAR_LABEL' in umap:
        label = str(meta_row[umap['SKY_FAR_LABEL']]).strip().upper()

    out = []
    for cname in ctx_names_local:
        key = str(cname).upper()

        if mode == 'sci':
            sci_key = f'SCI_{key}'
            if sci_key in umap:
                out.append(_safe_float(meta_row[umap[sci_key]]))
                continue

        if key in umap:
            out.append(_safe_float(meta_row[umap[key]]))
            continue

        skye_key = f'SKYE_{key}'
        skyw_key = f'SKYW_{key}'
        has_skye = skye_key in umap
        has_skyw = skyw_key in umap

        if has_skye and has_skyw and mode in ('near', 'far'):
            v_e = _safe_float(meta_row[umap[skye_key]])
            v_w = _safe_float(meta_row[umap[skyw_key]])
            out.append(v_w if label == 'SKYW' else v_e)
            continue

        raise KeyError(f"Missing context field for '{cname}' in META row")

    return np.asarray(out, dtype=np.float32)


def _infer_base_dir_for_reconstruction():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    if 'PALACE_DIR' in globals():
        try:
            p = Path(PALACE_DIR).resolve()
            candidates.extend([p, p.parent])
        except Exception:
            pass

    for cand in candidates:
        if (cand / 'palace' / 'PMD').exists() and (cand / 'Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt').exists():
            return cand

    raise FileNotFoundError('Could not infer reconstruction base_dir containing palace/PMD and solar reference file')


def predict_and_plot_three_fields(filename, row_index, predict_coef_from_context_fn, ctx_names_local):
    """Visual check helper.

    Parameters
    ----------
    predict_coef_from_context_fn : callable
        Function(ctx_row_phys) -> predicted coefficient vector.
    ctx_names_local : list[str]
        Context columns expected by the predictor.
    """
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    with fits.open(path) as hdul:
        if 'META' not in [h.name for h in hdul]:
            raise KeyError('Missing META extension')
        meta = Table(hdul['META'].data)

        i = int(row_index)
        if i < 0 or i >= len(meta):
            raise IndexError(f'row_index {i} out of range [0, {len(meta)-1}]')

        meta_row = meta[i]
        print('META row used for context generation:')
        _display_scrollable_table(meta[i:i + 1])

        wave_local = _read_ext_row(hdul, 'WAVE', i)
        lsf_row = _read_ext_row(hdul, 'LSF_SCI', i)
        flux_near = _read_ext_row(hdul, 'FLUX_SKY_NEAR', i)
        flux_far = _read_ext_row(hdul, 'FLUX_SKY_FAR', i)
        flux_sci = _read_ext_row(hdul, 'FLUX_SCI', i)

    ctx_near = _context_from_meta_row(meta_row, ctx_names_local, mode='near')
    ctx_far = _context_from_meta_row(meta_row, ctx_names_local, mode='far')
    ctx_sci = _context_from_meta_row(meta_row, ctx_names_local, mode='sci')

    coef_near = predict_coef_from_context_fn(ctx_near)
    coef_far = predict_coef_from_context_fn(ctx_far)
    coef_sci = predict_coef_from_context_fn(ctx_sci)

    base_dir_guess = _infer_base_dir_for_reconstruction()

    comps_near = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_near,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_far = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_far,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )
    comps_sci = reconstruct_component_spectra(
        wave=wave_local,
        coef=coef_sci,
        lsf_sigma=lsf_row / 2.35,
        n_spline_knots=25,
        base_dir=base_dir_guess,
    )

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=('Near Sky', 'Far Sky', 'Science Field'),
    )

    panel_data = [
        (1, flux_near, comps_near['total']),
        (2, flux_far, comps_far['total']),
        (3, flux_sci, comps_sci['total']),
    ]
    for row, flux_obs, flux_pred in panel_data:
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_obs * FACTOR,
                mode='lines',
                name='observed' if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color='#1f77b4', width=1.2),
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scattergl(
                x=wave_local,
                y=flux_pred,
                mode='lines',
                name='predicted reconstruction' if row == 1 else None,
                showlegend=(row == 1),
                line=dict(color='#d62728', width=1.2),
            ),
            row=row,
            col=1,
        )

    fig.update_yaxes(type='log', title_text='Flux', row=1, col=1)
    fig.update_yaxes(type='log', title_text='Flux', row=2, col=1)
    fig.update_yaxes(type='log', title_text='Flux', row=3, col=1)
    fig.update_xaxes(title_text='Wavelength [A]', row=3, col=1)
    fig.update_layout(
        template='plotly_white',
        height=980,
        title=f'Row {i}: predicted vs stored spectra (near/far/sci)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
    )
    fig.show()

    return {
        'file': str(path),
        'row_index': i,
        'wave': wave_local,
        'near': {'context': ctx_near, 'coef_pred': coef_near, 'flux_obs': flux_near, 'components': comps_near},
        'far': {'context': ctx_far, 'coef_pred': coef_far, 'flux_obs': flux_far, 'components': comps_far},
        'sci': {'context': ctx_sci, 'coef_pred': coef_sci, 'flux_obs': flux_sci, 'components': comps_sci},
    }

In [ ]:
# Starter data load for coefficient prediction experiments
context_cols = [
    'alt',
    'moon_sep',
    'moon_alt',
    'sun_alt',
    'moon_illum',
    'airmass',
    'obstime_mjd',
    'obstime_mjd_z',
    'obstime_year_sin',
    'obstime_year_cos',
    'obstime_day_sin',
    'obstime_day_cos',
]

triplet = build_triplet_coef_dataset(
    input_fits_path='lvmsframe_median_stack_1.2.1_meta_only.fits',
    sky_near_decomp_fits_path='lvmsframe_median_stack_1.2.1_sky1_meta_coef.fits',
    sky_far_decomp_fits_path='lvmsframe_median_stack_1.2.1_sky2_meta_coef.fits',
    sci_decomp_fits_path='lvmsframe_median_stack_1.2.1_sci_meta_coef.fits',
    context_columns=context_cols,
    return_chi2=True,
)

print('Shapes:')
print('  coef_near', triplet['coef_near'].shape)
print('  coef_far ', triplet['coef_far'].shape)
print('  coef_sci ', triplet['coef_sci'].shape)
print('  ctx_near ', triplet['ctx_near'].shape)
print('  ctx_far  ', triplet['ctx_far'].shape)
print('  ctx_sci  ', triplet['ctx_sci'].shape)

## Filtering And Coefficient Learning

Apply the same robust row filtering used in the original notebook, then train a compact coefficient-prediction model that maps simultaneous near/far sky coefficients and geometry context to science-location coefficients.

In [ ]:
# Filtering borrowed from the original notebook workflow, adapted to triplets
import plotly.express as px


def _coef_to_model_space(coef):
    return np.sqrt(np.clip(np.asarray(coef, dtype=np.float32), 0.0, None)).astype(np.float32)


def _coef_from_model_space(coef_model):
    coef_model = np.clip(np.asarray(coef_model, dtype=np.float32), 0.0, None)
    return np.square(coef_model).astype(np.float32)


def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep

    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep

    return keep


def apply_triplet_filters(
    triplet_data,
    thin_every_n=2,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds=None,
    kappa=6.0,
    kappa_iter=3,
):
    if hard_coef_bounds is None:
        hard_coef_bounds = {"feo": (0.0, 0.01), "atom_k": (0.0, 0.01)}

    coef_names_local = [str(n) for n in triplet_data["coef_names"]]
    coef_name_l = [n.lower() for n in coef_names_local]

    coef_near = np.asarray(triplet_data["coef_near"], dtype=np.float32)
    coef_far = np.asarray(triplet_data["coef_far"], dtype=np.float32)
    coef_sci = np.asarray(triplet_data["coef_sci"], dtype=np.float32)
    ctx_near = np.asarray(triplet_data["ctx_near"], dtype=np.float32)
    ctx_far = np.asarray(triplet_data["ctx_far"], dtype=np.float32)
    ctx_sci = np.asarray(triplet_data["ctx_sci"], dtype=np.float32)

    n0 = coef_near.shape[0]
    keep = np.ones(n0, dtype=bool)

    # Optional thinning, same as the original notebook.
    if int(thin_every_n) > 1:
        thin_mask = np.zeros(n0, dtype=bool)
        thin_mask[:: int(thin_every_n)] = True
        keep &= thin_mask
        print(f"Pre-chi2 thinning: every {int(thin_every_n)} row kept -> n_rows={thin_mask.sum()}")
    else:
        print(f"Pre-chi2 thinning disabled: n_rows={n0}")

    # Physical context sanity check:
    # keep only rows with non-negative telescope altitude ('alt') and
    # non-negative airmass in near/far/sci contexts.
    # Note: sun altitude ('sun_alt') and moon altitude ('moon_alt') are intentionally NOT constrained here.
    ctx_names_l = [str(n).strip().lower() for n in triplet_data["ctx_names"]]
    alt_idx = ctx_names_l.index("alt") if "alt" in ctx_names_l else None
    airmass_idx = ctx_names_l.index("airmass") if "airmass" in ctx_names_l else None
    if "moon_alt" in ctx_names_l:
        print("Moon-altitude sanity check: moon_alt is available but intentionally NOT used for filtering.")

    if alt_idx is None:
        print("Altitude sanity filter: context column 'alt' not found; skipping altitude >= 0 check.")
    if airmass_idx is None:
        print("Airmass sanity filter: context column 'airmass' not found; skipping airmass >= 0 check.")

    if alt_idx is not None or airmass_idx is not None:
        physical_mask = np.ones(n0, dtype=bool)

        if alt_idx is not None:
            alt_ok = (
                np.isfinite(ctx_near[:, alt_idx]) & (ctx_near[:, alt_idx] >= 0.0)
                & np.isfinite(ctx_far[:, alt_idx]) & (ctx_far[:, alt_idx] >= 0.0)
                & np.isfinite(ctx_sci[:, alt_idx]) & (ctx_sci[:, alt_idx] >= 0.0)
            )
            physical_mask &= alt_ok
            print(
                f"Altitude sanity filter (alt >= 0 in near/far/sci): kept {alt_ok.sum()}/{alt_ok.size} "
                f"({100.0 * alt_ok.mean():.1f}%)"
            )

        if airmass_idx is not None:
            airmass_ok = (
                np.isfinite(ctx_near[:, airmass_idx]) & (ctx_near[:, airmass_idx] >= 0.0)
                & np.isfinite(ctx_far[:, airmass_idx]) & (ctx_far[:, airmass_idx] >= 0.0)
                & np.isfinite(ctx_sci[:, airmass_idx]) & (ctx_sci[:, airmass_idx] >= 0.0)
            )
            physical_mask &= airmass_ok
            print(
                f"Airmass sanity filter (airmass >= 0 in near/far/sci): kept {airmass_ok.sum()}/{airmass_ok.size} "
                f"({100.0 * airmass_ok.mean():.1f}%)"
            )

        keep &= physical_mask
        print(
            f"Combined physical context filter: kept {physical_mask.sum()}/{len(physical_mask)} "
            f"({100.0 * physical_mask.mean():.1f}%)"
        )

    # Combined chi2 gating across near/far/sci when available.
    if all(k in triplet_data for k in ("chi2_near", "chi2_far", "chi2_sci")):
        chi2_stack = np.column_stack(
            [
                np.asarray(triplet_data["chi2_near"], dtype=np.float64),
                np.asarray(triplet_data["chi2_far"], dtype=np.float64),
                np.asarray(triplet_data["chi2_sci"], dtype=np.float64),
            ]
        )
        chi2_combined = np.nanmax(chi2_stack, axis=1)
        chi2_finite = chi2_combined[np.isfinite(chi2_combined)]
        chi2_hi = np.nanpercentile(chi2_finite, chi2_qmax)
        chi2_upper = min(float(chi2_max), float(chi2_hi)) if chi2_max is not None else float(chi2_hi)
        chi2_mask = np.isfinite(chi2_combined) & (chi2_combined >= chi2_min) & (chi2_combined <= chi2_upper)
        keep &= chi2_mask
        print(
            f"Triplet chi2 filter: min={chi2_min:.3g}, qmax={chi2_qmax:.1f}%=>{chi2_hi:.3g}, "
            f"upper={chi2_upper:.3g} | keep={chi2_mask.sum()}/{len(chi2_mask)} ({100.0*chi2_mask.mean():.1f}%)"
        )
        fig_chi2_triplet = px.histogram(
            x=chi2_combined[keep],
            nbins=80,
            title="Triplet combined reduced chi2 distribution (rows used for training)",
            labels={"x": "max(reduced chi2 near/far/sci)", "y": "count"},
        )
        fig_chi2_triplet.update_layout(template="plotly_white", bargap=0.03)
        fig_chi2_triplet.show()
    else:
        print("Triplet chi2 columns not present; chi2 filtering skipped.")

    # Manual hard coefficient bounds applied to all three fields.
    for cname, (lo, hi) in hard_coef_bounds.items():
        idxs = np.where(np.array(coef_name_l) == str(cname).lower())[0]
        if idxs.size == 0:
            print(f"Manual hard clip: coefficient {cname} not found; skipping.")
            continue

        j = int(idxs[0])
        within = (
            np.isfinite(coef_near[:, j]) & (coef_near[:, j] >= lo) & (coef_near[:, j] <= hi)
            & np.isfinite(coef_far[:, j]) & (coef_far[:, j] >= lo) & (coef_far[:, j] <= hi)
            & np.isfinite(coef_sci[:, j]) & (coef_sci[:, j] >= lo) & (coef_sci[:, j] <= hi)
        )
        keep &= within
        print(
            f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} ({100.0 * within.mean():.1f}%)"
        )

    # Kappa-sigma clipping in concatenated coefficient space.
    coef_concat = np.hstack([coef_near, coef_far, coef_sci]).astype(np.float32)
    kappa_mask = _kappa_sigma_row_mask(coef_concat, kappa=float(kappa), n_iter=int(kappa_iter))
    keep &= kappa_mask
    print(
        f"Kappa-sigma filter (kappa={kappa:.1f}): kept {kappa_mask.sum()}/{len(kappa_mask)} ({100.0 * kappa_mask.mean():.1f}%)"
    )

    if keep.sum() == 0:
        raise RuntimeError("Filtering removed all rows; relax thresholds.")

    out = {
        "coef_near": coef_near[keep],
        "coef_far": coef_far[keep],
        "coef_sci": coef_sci[keep],
        "ctx_near": ctx_near[keep],
        "ctx_far": ctx_far[keep],
        "ctx_sci": ctx_sci[keep],
        "coef_names": coef_names_local,
        "ctx_names": list(triplet_data["ctx_names"]),
        "mask": keep,
    }
    for k in ("chi2_near", "chi2_far", "chi2_sci"):
        if k in triplet_data:
            out[k] = np.asarray(triplet_data[k])[keep]

    print(
        f"Filtered triplet shapes: near={out['coef_near'].shape}, far={out['coef_far'].shape}, "
        f"sci={out['coef_sci'].shape}"
    )

    # Context-parameter histograms split by sky near, sky far, and science,
    # now using the final post-filter row set.
    ctx_sets_df = pd.concat(
        [
            pd.DataFrame(out["ctx_near"], columns=out["ctx_names"]).assign(sky_field="sky_near"),
            pd.DataFrame(out["ctx_far"], columns=out["ctx_names"]).assign(sky_field="sky_far"),
            pd.DataFrame(out["ctx_sci"], columns=out["ctx_names"]).assign(sky_field="science"),
        ],
        ignore_index=True,
    )
    ctx_long = ctx_sets_df.melt(
        id_vars="sky_field",
        var_name="context_param",
        value_name="context_value",
    )
    fig_ctx_hist = px.histogram(
        ctx_long,
        x="context_value",
        facet_row="sky_field",
        facet_col="context_param",
        nbins=70,
        title="Context parameter distributions by field (rows used after all filters)",
        labels={
            "context_value": "value",
            "count": "count",
            "context_param": "context",
            "sky_field": "field",
        },
    )

    # Force independent histogram binning in each subplot.
    for i, tr in enumerate(fig_ctx_hist.data):
        tr.update(bingroup=f"ctx_field_{i}", autobinx=True, xbins=dict())

    fig_ctx_hist.for_each_annotation(
        lambda a: a.update(
            text=(
                a.text
                .replace("context_param=", "")
                .replace("context=", "")
                .replace("sky_field=", "")
                .replace("field=", "")
            )
        )
    )
    fig_ctx_hist.update_xaxes(matches=None)
    fig_ctx_hist.update_yaxes(matches=None)

    # Ensure x-axis tick labels and marks are visible on every subplot.
    fig_ctx_hist.update_xaxes(showticklabels=True, ticks="outside", ticklen=4, showline=True, mirror=True)
    fig_ctx_hist.update_layout(template="plotly_white", bargap=0.03, height=880)
    fig_ctx_hist.show()

    return out


filtered_triplet = apply_triplet_filters(
    triplet,
    thin_every_n=1,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 0.01), "atom_k": (0.0, 0.01)},
    kappa=6.0,
    kappa_iter=3,
)


In [ ]:
# Shared ML utilities for coefficient prediction models
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


class RobustScaler:
    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0).astype(np.float32)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return x * self.scale_ + self.med_


def _set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx


def split_indices_by_sorted_time(time_values, train_frac=0.8, val_frac=0.1):
    t = np.asarray(time_values, dtype=np.float64).reshape(-1)
    if t.size == 0:
        raise ValueError('time_values is empty')
    if not np.isfinite(t).all():
        raise ValueError('time_values contains non-finite values')

    idx = np.argsort(t, kind='mergesort')
    n = idx.size
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx


def _make_loader(*arrays, batch_size=256, shuffle=False):
    tensors = [torch.from_numpy(np.asarray(a, dtype=np.float32)) for a in arrays]
    ds = TensorDataset(*tensors)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## Deep Group-Head MLP Coefficient Model

This notebook uses a deterministic deep group-head MLP to predict science-fiber decomposition coefficients from near/far coefficients and context metadata.

### Architecture And Structure
The model is split into:

1. Shared trunk MLP:
- input: concatenated normalized feature vector
- layers: configurable `trunk_dims` (default style: 1024 -> 512 -> 256)
- activation: GELU after each hidden linear layer
- output: shared embedding vector used by all output heads

2. Group-specific heads:
- each coefficient family has its own small MLP head
- each head maps the shared embedding to the coefficients of that family
- families are inferred from coefficient names: `moon`, `oh`, `diffuse`, `atomic`

### Input Mapping
For each row, inputs are built as:

- near coefficients: `coef_near`
- far coefficients: `coef_far`
- context vectors: `ctx_near`, `ctx_far`, `ctx_sci`
- relative context feature: `delta_ctx = ctx_sci - 0.5 * (ctx_near + ctx_far)`

Normalization pipeline:

1. map physical coefficients to model space via `_coef_to_model_space`
2. robust-scale coefficients and contexts with train-fit `RobustScaler`
3. clip normalized values to `[-25, 25]`

Final model input vector:

`X = [near_n, far_n, delta_ctx_n, sci_ctx_n]`

### Embedding
The trunk output is the latent embedding of each sample:

- shared across all heads
- captures cross-family coupling before head-specific decoding
- reused later for embedding diagnostics in notebook cells

### Heads And Outputs
Each head predicts only its own coefficient indices.
All head outputs are stitched back into full SCI coefficient order.

Output post-processing:

1. inverse robust-scaling
2. inverse model-space mapping via `_coef_from_model_space`
3. non-negativity clamp (`>= 0`)

Returned output is a full predicted SCI coefficient vector per row in physical space.

### Loss Function
Training uses weighted multi-head SmoothL1 loss:

- per group loss: `SmoothL1(pred_group, target_group)`
- group weight: `1 / sqrt(group_size)`
- final loss: average across active groups

This reduces domination by large groups while keeping robustness to outliers.

### Priors And Post-MLP Correction
The default predictor path is:

1. base deep group-head MLP prediction
2. Ridge residual correction in coefficient space
3. Moon spline smoothness prior (inference-time)

Moon prior details:

- applied only to `moon_bs` coefficients
- uses second-difference penalty matrix `D2`
- candidate lambda selected from `lambda_grid`
- chosen lambda matches roughness to a near/far-derived target

Lambda is a dimensionless regularization weight in coefficient-index space.

### Classes And Functions To Know
- class `DeepGroupHeadMLP`: trunk + heads model definition
- `_build_group_indices`: map coefficient names to output groups
- `_group_loss`: weighted multi-head SmoothL1 objective
- `train_coeff_prediction_group_mlp`: training routine
- `predict_sci_coefficients_group_mlp`: deterministic base predictor
- `predict_sci_coefficients_default`: default inference API (base + residual + prior)

In [ ]:
# Deep group-head MLP model and helpers for conditional coefficient prediction
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def _build_group_indices(coef_names):
    """Build semantic output groups from coefficient names.

    Parameters
    ----------
    coef_names : sequence[str]
        Coefficient names in SCI output order.

    Returns
    -------
    dict[str, np.ndarray]
        Group name -> output index array.
    """
    coef_names_l = [str(n).lower() for n in coef_names]
    groups = {
        "moon": np.asarray([i for i, n in enumerate(coef_names_l) if n.startswith("moon_bs")], dtype=int),
        "oh": np.asarray([i for i, n in enumerate(coef_names_l) if n.startswith("oh")], dtype=int),
        "diffuse": np.asarray(
            [
                i
                for i, n in enumerate(coef_names_l)
                if (n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or ("diffuse" in n) or ("continuum" in n))
            ],
            dtype=int,
        ),
        "atomic": np.asarray([i for i, n in enumerate(coef_names_l) if n.startswith("atom")], dtype=int),
    }

    # Drop empty groups so loss and heads only cover valid families.
    groups = {k: v for k, v in groups.items() if len(v) > 0}
    if len(groups) == 0:
        raise RuntimeError("No coefficient groups were detected from coefficient names")

    print("Coefficient group counts:", ", ".join([f"{k}={len(v)}" for k, v in groups.items()]))
    return groups


class DeepGroupHeadMLP(nn.Module):
    """Shared-trunk multi-head MLP for SCI coefficient regression.

    Notes
    -----
    - The trunk learns a shared embedding from all input features.
    - Each coefficient family has its own output head.
    - `forward` returns a dict keyed by group name.
    """

    def __init__(self, in_dim, group_indices, trunk_dims=(1024, 512, 256), head_dim=256):
        """Initialize trunk and per-group heads.

        Parameters
        ----------
        in_dim : int
            Input feature dimension.
        group_indices : dict[str, np.ndarray]
            Output index mapping per group.
        trunk_dims : tuple[int, ...]
            Hidden sizes for shared trunk.
        head_dim : int
            Hidden size for each group head.
        """
        super().__init__()
        self.group_names = list(group_indices.keys())
        self.group_indices = {k: np.asarray(v, dtype=np.int64) for k, v in group_indices.items()}

        layers = []
        last = int(in_dim)
        for d in trunk_dims:
            layers.append(nn.Linear(last, int(d)))
            layers.append(nn.GELU())
            last = int(d)
        self.trunk = nn.Sequential(*layers)

        self.heads = nn.ModuleDict(
            {
                g: nn.Sequential(
                    nn.Linear(last, int(head_dim)),
                    nn.GELU(),
                    nn.Linear(int(head_dim), int(len(self.group_indices[g]))),
                )
                for g in self.group_names
            }
        )

    def forward(self, x):
        """Run trunk embedding and all heads.

        Parameters
        ----------
        x : torch.Tensor
            Shape (batch, in_dim).

        Returns
        -------
        dict[str, torch.Tensor]
            Per-group predictions.
        """
        h = self.trunk(x)
        return {g: self.heads[g](h) for g in self.group_names}


def _group_loss(pred, y_true, group_idx_t):
    """Compute weighted multi-head SmoothL1 loss.

    Uses weight `1/sqrt(group_size)` to balance groups.
    """
    loss = torch.tensor(0.0, device=y_true.device)
    for g, idx_t in group_idx_t.items():
        weight = 1.0 / np.sqrt(max(int(idx_t.numel()), 1))
        loss = loss + float(weight) * F.smooth_l1_loss(pred[g], y_true[:, idx_t])
    return loss / max(len(group_idx_t), 1)


def train_coeff_prediction_group_mlp(
    filtered,
    n_epochs=40,
    batch_size=256,
    lr=1e-3,
    trunk_dims=(1024, 512, 256),
    head_dim=256,
    weight_decay=1e-4,
    grad_clip=1.0,
    patience=8,
    split_mode="time",
    seed=42,
):
    """Train deep group-head MLP on filtered triplet dataset.

    Input features are `[near_n, far_n, delta_ctx_n, sci_ctx_n]` and targets are
    normalized SCI coefficients in model space.
    """
    _set_reproducibility(seed)

    coef_near = np.asarray(filtered["coef_near"], dtype=np.float32)
    coef_far = np.asarray(filtered["coef_far"], dtype=np.float32)
    coef_sci = np.asarray(filtered["coef_sci"], dtype=np.float32)
    ctx_near = np.asarray(filtered["ctx_near"], dtype=np.float32)
    ctx_far = np.asarray(filtered["ctx_far"], dtype=np.float32)
    ctx_sci = np.asarray(filtered["ctx_sci"], dtype=np.float32)

    n = coef_near.shape[0]
    split_mode_local = str(split_mode).strip().lower()
    ctx_names_local = [str(x) for x in filtered.get("ctx_names", [])]

    # Prefer time-ordered split when obstime_mjd is available.
    if split_mode_local == "time" and "obstime_mjd" in ctx_names_local:
        time_col = ctx_names_local.index("obstime_mjd")
        train_idx, val_idx, test_idx = split_indices_by_sorted_time(ctx_sci[:, time_col])
        print("Using time-ordered split (obstime_mjd) for train/val/test.")
    else:
        if split_mode_local == "time":
            print("split_mode='time' requested but obstime_mjd not found; falling back to random split.")
        train_idx, val_idx, test_idx = split_indices(n, seed=seed)

    # Map physical coefficients to model space before robust normalization.
    near_model = _coef_to_model_space(coef_near)
    far_model = _coef_to_model_space(coef_far)
    sci_model = _coef_to_model_space(coef_sci)

    coef_scaler = RobustScaler().fit(
        np.vstack([near_model[train_idx], far_model[train_idx], sci_model[train_idx]])
    )
    ctx_scaler = RobustScaler().fit(
        np.vstack([ctx_near[train_idx], ctx_far[train_idx], ctx_sci[train_idx]])
    )

    near_n = np.clip(coef_scaler.transform(near_model), -25.0, 25.0).astype(np.float32)
    far_n = np.clip(coef_scaler.transform(far_model), -25.0, 25.0).astype(np.float32)
    sci_n = np.clip(coef_scaler.transform(sci_model), -25.0, 25.0).astype(np.float32)

    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)

    # Relative science context signal.
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    X_all = np.hstack([near_n, far_n, delta_ctx_n, sci_ctx_n]).astype(np.float32)
    Y_all = sci_n.astype(np.float32)

    X_tr, X_va = X_all[train_idx], X_all[val_idx]
    y_tr, y_va = Y_all[train_idx], Y_all[val_idx]

    tr_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=int(batch_size),
        shuffle=True,
    )
    va_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va)),
        batch_size=512,
        shuffle=False,
    )

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    group_indices = _build_group_indices(filtered["coef_names"])
    group_idx_t = {g: torch.as_tensor(idx, dtype=torch.long, device=device) for g, idx in group_indices.items()}

    model = DeepGroupHeadMLP(
        X_all.shape[1],
        group_indices=group_indices,
        trunk_dims=trunk_dims,
        head_dim=int(head_dim),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=float(lr), weight_decay=float(weight_decay))

    history = []
    best_val = np.inf
    best_epoch = -1
    best_state = None
    stale = 0

    for ep in range(1, int(n_epochs) + 1):
        model.train()
        tr_loss = 0.0
        tr_n = 0
        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)
            loss = _group_loss(pred, yb, group_idx_t)
            if not torch.isfinite(loss):
                continue

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()

            tr_loss += float(loss.item())
            tr_n += 1

        model.eval()
        va_loss = 0.0
        va_n = 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                pred = model(xb)
                loss = _group_loss(pred, yb, group_idx_t)
                if not torch.isfinite(loss):
                    continue
                va_loss += float(loss.item())
                va_n += 1

        tr_loss_m = tr_loss / max(tr_n, 1)
        va_loss_m = va_loss / max(va_n, 1)
        history.append({"epoch": ep, "train_loss": tr_loss_m, "val_loss": va_loss_m})

        if ep == 1 or ep % 10 == 0:
            print(f"[deep-group-mlp] epoch={ep:03d} train={tr_loss_m:.6f} val={va_loss_m:.6f}")

        if va_loss_m < best_val:
            best_val = va_loss_m
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1

        if stale >= int(patience):
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "model": model,
        "device": device,
        "coef_scaler": coef_scaler,
        "ctx_scaler": ctx_scaler,
        "history": history,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "coef_names": list(filtered["coef_names"]),
        "ctx_names": list(filtered["ctx_names"]),
        "group_indices": {k: v.copy() for k, v in group_indices.items()},
        "trunk_dims": tuple(int(d) for d in trunk_dims),
        "head_dim": int(head_dim),
        "split_mode": split_mode_local,
    }


def predict_sci_coefficients_group_mlp(artifacts, coef_near_phys, coef_far_phys, ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Predict SCI coefficients with trained deep group-head MLP.

    Returns non-negative physical coefficients, batched.
    """
    model = artifacts["model"]
    device = artifacts["device"]
    coef_scaler = artifacts["coef_scaler"]
    ctx_scaler = artifacts["ctx_scaler"]
    group_indices = artifacts["group_indices"]

    coef_near_phys = np.asarray(coef_near_phys, dtype=np.float32)
    coef_far_phys = np.asarray(coef_far_phys, dtype=np.float32)
    ctx_near_phys = np.asarray(ctx_near_phys, dtype=np.float32)
    ctx_far_phys = np.asarray(ctx_far_phys, dtype=np.float32)
    ctx_sci_phys = np.asarray(ctx_sci_phys, dtype=np.float32)

    if coef_near_phys.ndim == 1:
        coef_near_phys = coef_near_phys[None, :]
    if coef_far_phys.ndim == 1:
        coef_far_phys = coef_far_phys[None, :]
    if ctx_near_phys.ndim == 1:
        ctx_near_phys = ctx_near_phys[None, :]
    if ctx_far_phys.ndim == 1:
        ctx_far_phys = ctx_far_phys[None, :]
    if ctx_sci_phys.ndim == 1:
        ctx_sci_phys = ctx_sci_phys[None, :]

    near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near_phys)), -25.0, 25.0).astype(np.float32)
    far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far_phys)), -25.0, 25.0).astype(np.float32)
    near_ctx_n = np.clip(ctx_scaler.transform(ctx_near_phys), -25.0, 25.0).astype(np.float32)
    far_ctx_n = np.clip(ctx_scaler.transform(ctx_far_phys), -25.0, 25.0).astype(np.float32)
    sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci_phys), -25.0, 25.0).astype(np.float32)
    delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

    X = np.hstack([near_n, far_n, delta_ctx_n, sci_ctx_n]).astype(np.float32)

    out_n = np.zeros((X.shape[0], near_n.shape[1]), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for s in range(0, X.shape[0], 512):
            e = min(s + 512, X.shape[0])
            xb = torch.from_numpy(X[s:e]).to(device)
            pred = model(xb)
            for g, idx in group_indices.items():
                out_n[s:e, idx] = pred[g].cpu().numpy().astype(np.float32)

    out_model = coef_scaler.inverse_transform(out_n)
    out_phys = _coef_from_model_space(out_model)
    return np.clip(out_phys.astype(np.float32), 0.0, None)


# Compatibility wrapper names so downstream notebook cells can run unchanged.
def train_coeff_prediction_cvae(filtered, **kwargs):
    """Compatibility shim for legacy cVAE training call sites."""
    mapped = {
        "n_epochs": int(kwargs.get("n_epochs", 40)),
        "batch_size": int(kwargs.get("batch_size", 256)),
        "lr": float(kwargs.get("lr", 1e-3)),
        "seed": int(kwargs.get("seed", 42)),
        "grad_clip": float(kwargs.get("grad_clip", 1.0)),
    }
    if "patience" in kwargs:
        mapped["patience"] = int(kwargs["patience"])
    if "weight_decay" in kwargs:
        mapped["weight_decay"] = float(kwargs["weight_decay"])
    if "split_mode" in kwargs:
        mapped["split_mode"] = str(kwargs["split_mode"])
    return train_coeff_prediction_group_mlp(filtered, **mapped)


def predict_sci_coefficients_cvae(
    cvae_artifacts,
    coef_near_phys,
    coef_far_phys,
    ctx_near_phys,
    ctx_far_phys,
    ctx_sci_phys,
    deterministic=True,
    n_samples=16,
    seed=42,
    return_samples=False,
):
    """Compatibility shim for legacy cVAE inference call sites.

    This notebook variant is deterministic-only.
    """
    if not deterministic or return_samples:
        raise ValueError("Deep group-head MLP is deterministic only in this notebook")
    return predict_sci_coefficients_group_mlp(
        cvae_artifacts,
        coef_near_phys=coef_near_phys,
        coef_far_phys=coef_far_phys,
        ctx_near_phys=ctx_near_phys,
        ctx_far_phys=ctx_far_phys,
        ctx_sci_phys=ctx_sci_phys,
    )

In [ ]:
# Clear stale variables from earlier time/no-time ablation experiments.
# Keeps the baseline run reproducible and avoids accidental reuse from kernel state.
_ablation_stale_vars = [
    "compare_time_no_time_df",
    "compare_time_no_time_holdout_df",
    "res_time",
    "res_no_time",
    "res_time_holdout",
    "res_no_time_holdout",
    "cfg_time",
    "cfg",
    "ctx_names_all",
    "keep_idx",
    "filtered_no_time",
    "time_feature_names",
    "d_rmse",
    "d_mae",
    "d_rmse_h",
    "d_mae_h",
]

for _name in _ablation_stale_vars:
    if _name in globals():
        del globals()[_name]

print("Cleared stale ablation variables:", ", ".join(_ablation_stale_vars))
del _name, _ablation_stale_vars

In [ ]:
# Train the baseline deep group-head MLP (time-enabled context) and summarize deterministic test metrics
required = [
    "filtered_triplet",
    "train_coeff_prediction_group_mlp",
    "predict_sci_coefficients_group_mlp",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run prerequisite cells first. Missing: " + ", ".join(missing))

# Baseline configuration: includes time features in context and uses time-ordered split.
deep_group_config = {
    "name": "deep_group_mlp_time_baseline",
    "n_epochs": 30,
    "batch_size": 256,
    "lr": 6e-4,
    "trunk_dims": (768, 384, 192),
    "head_dim": 256,
    "weight_decay": 1e-4,
    "patience": 8,
    "split_mode": "time",
}

print("=== Training deep group-head MLP baseline config ===")
print(deep_group_config)

cvae_artifacts = train_coeff_prediction_group_mlp(
    filtered_triplet,
    n_epochs=int(deep_group_config["n_epochs"]),
    batch_size=int(deep_group_config["batch_size"]),
    lr=float(deep_group_config["lr"]),
    trunk_dims=tuple(int(v) for v in deep_group_config["trunk_dims"]),
    head_dim=int(deep_group_config["head_dim"]),
    weight_decay=float(deep_group_config["weight_decay"]),
    grad_clip=1.0,
    patience=int(deep_group_config["patience"]),
    split_mode=str(deep_group_config.get("split_mode", "time")),
    seed=42,
)

print(
    f"Selected deep group-head config: {deep_group_config['name']} | "
    f"best epoch: {cvae_artifacts['best_epoch']} | "
    f"best val score: {cvae_artifacts['best_val_loss']:.6f}"
)

test_idx_cvae = np.asarray(cvae_artifacts["test_idx"], dtype=int)
coef_true_cvae = np.asarray(filtered_triplet["coef_sci"][test_idx_cvae], dtype=np.float32)

coef_pred_cvae_det = predict_sci_coefficients_group_mlp(
    cvae_artifacts,
    coef_near_phys=filtered_triplet["coef_near"][test_idx_cvae],
    coef_far_phys=filtered_triplet["coef_far"][test_idx_cvae],
    ctx_near_phys=filtered_triplet["ctx_near"][test_idx_cvae],
    ctx_far_phys=filtered_triplet["ctx_far"][test_idx_cvae],
    ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx_cvae],
)


def _metric_row(y_true, y_pred, model_name):
    """Compute aggregate per-coefficient RMSE/MAE/correlation diagnostics."""
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    corr = []
    for j in range(y_true.shape[1]):
        x = y_true[:, j]
        y = y_pred[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corr.append(np.nan)
        else:
            corr.append(float(np.corrcoef(x, y)[0, 1]))
    corr = np.asarray(corr)
    return {
        "model": model_name,
        "mean_rmse": float(np.nanmean(rmse)),
        "median_rmse": float(np.nanmedian(rmse)),
        "mean_mae": float(np.nanmean(mae)),
        "mean_corr": float(np.nanmean(corr)),
        "median_corr": float(np.nanmedian(corr)),
    }


cmp_df = pd.DataFrame([
    _metric_row(coef_true_cvae, coef_pred_cvae_det, "deep_group_mlp_time_baseline")
])

print("\nBaseline deep group-head coefficient prediction summary on test split:")
print(cmp_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

In [ ]:
# Post deep-group MLP residual corrector and default predictor interface.
#
# Model structure used by downstream cells:
# 1) Deep group-head MLP base prediction.
# 2) Ridge residual correction in coefficient space.
# 3) Smoothness prior on Moon B-spline coefficients.
#
# The helper `predict_sci_coefficients_default(...)` is the single default path.

from sklearn.linear_model import Ridge

required = ["cvae_artifacts", "filtered_triplet", "predict_sci_coefficients_group_mlp"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run deep-group MLP training cell first. Missing: " + ", ".join(missing))

train_idx = np.asarray(cvae_artifacts["train_idx"], dtype=int)
val_idx = np.asarray(cvae_artifacts["val_idx"], dtype=int)
test_idx = np.asarray(cvae_artifacts["test_idx"], dtype=int)

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float32)
ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)


def _base_pred_for_idx(idx):
    """Deterministic deep group-head MLP prediction for a row index array."""
    return predict_sci_coefficients_group_mlp(
        cvae_artifacts,
        coef_near_phys=coef_near_all[idx],
        coef_far_phys=coef_far_all[idx],
        ctx_near_phys=ctx_near_all[idx],
        ctx_far_phys=ctx_far_all[idx],
        ctx_sci_phys=ctx_sci_all[idx],
    ).astype(np.float32)


def _build_features(idx, base_pred):
    """Feature block for residual model: base pred + inputs + context deltas."""
    x_near = coef_near_all[idx]
    x_far = coef_far_all[idx]
    x_scin = ctx_sci_all[idx]
    x_dctx = x_scin - 0.5 * (ctx_near_all[idx] + ctx_far_all[idx])
    return np.hstack([base_pred, x_near, x_far, x_dctx, x_scin]).astype(np.float32)


def _metric_row(y_true, y_pred, model_name):
    """Summarize per-coefficient error statistics for a prediction matrix."""
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    corr = []
    for j in range(y_true.shape[1]):
        x = y_true[:, j]
        y = y_pred[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corr.append(np.nan)
        else:
            corr.append(float(np.corrcoef(x, y)[0, 1]))
    corr = np.asarray(corr)
    return {
        "model": model_name,
        "mean_rmse": float(np.nanmean(rmse)),
        "median_rmse": float(np.nanmedian(rmse)),
        "mean_mae": float(np.nanmean(mae)),
        "mean_corr": float(np.nanmean(corr)),
        "median_corr": float(np.nanmedian(corr)),
    }


# Fit residual corrector using train split and select alpha on validation split.
base_tr = _base_pred_for_idx(train_idx)
base_va = _base_pred_for_idx(val_idx)
base_te = _base_pred_for_idx(test_idx)

y_tr = coef_sci_all[train_idx]
y_va = coef_sci_all[val_idx]
y_te = coef_sci_all[test_idx]

X_tr = _build_features(train_idx, base_tr)
X_va = _build_features(val_idx, base_va)
X_te = _build_features(test_idx, base_te)

res_tr = (y_tr - base_tr).astype(np.float32)

alphas = [0.05, 0.1, 0.3, 1.0, 3.0, 10.0]
best = {"alpha": None, "score": np.inf, "model": None}
for alpha in alphas:
    reg = Ridge(alpha=float(alpha), fit_intercept=True, random_state=42)
    reg.fit(X_tr, res_tr)
    pred_va = np.clip(base_va + reg.predict(X_va).astype(np.float32), 0.0, None)
    score = float(np.mean((pred_va - y_va) ** 2))
    if score < best["score"]:
        best = {"alpha": float(alpha), "score": score, "model": reg}

coef_residual_corrector = best["model"]
coef_residual_corrector_alpha = best["alpha"]

coef_pred_cvae_corr = np.clip(base_te + coef_residual_corrector.predict(X_te).astype(np.float32), 0.0, None)

cmp_post_df = pd.DataFrame([
    _metric_row(y_te, base_te, "deep_group_mlp"),
    _metric_row(y_te, coef_pred_cvae_corr, "deep_group_mlp_plus_ridge"),
])

print(f"Selected residual-corrector alpha: {coef_residual_corrector_alpha}")
print("\nTest split coefficient metrics (before/after residual correction):")
print(cmp_post_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))


# Smoothness-prior settings for Moon B-spline coefficients.
# lambda is a dimensionless regularization weight in (I + lambda * D2^T D2).
MOON_SPLINE_PRIOR_CONFIG = {
    "enabled": True,
    "lambda_grid": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "target_roughness_scale": 0.90,
    "min_lambda": 0.03,
    "eps": 1e-12,
}


def _moon_bs_indices_from_names(coef_names_local):
    """Return index positions of Moon B-spline coefficients."""
    return np.asarray(
        [i for i, n in enumerate(coef_names_local) if str(n).strip().lower().startswith("moon_bs")],
        dtype=int,
    )


def _row_spline_roughness(vals):
    """Compute RMS second-difference roughness for one coefficient vector."""
    v = np.asarray(vals, dtype=np.float64)
    if v.size < 3:
        return 0.0
    d2 = np.diff(v, n=2)
    return float(np.sqrt(np.mean(d2 ** 2)))


def _build_d2_matrix(n):
    """Construct second-difference operator matrix of size (n-2, n)."""
    d2 = np.zeros((n - 2, n), dtype=np.float64)
    for i in range(n - 2):
        d2[i, i:i + 3] = (1.0, -2.0, 1.0)
    return d2


def _smooth_nonneg_with_lambda(vals, lam, d2=None):
    """Apply non-negative Tikhonov smoothing with D2 penalty and weight lambda."""
    v = np.asarray(vals, dtype=np.float64)
    n = v.size
    if n < 3 or lam <= 0:
        return np.clip(v, 0.0, None)
    if d2 is None:
        d2 = _build_d2_matrix(n)
    a = np.eye(n, dtype=np.float64) + float(lam) * (d2.T @ d2)
    v_s = np.linalg.solve(a, v)
    return np.clip(v_s, 0.0, None)


def _apply_moon_spline_prior_batch(coef_pred, coef_near, coef_far, coef_names_local, prior_cfg):
    """Select and apply Moon spline smoothing strength per row from a lambda grid."""
    out = np.asarray(coef_pred, dtype=np.float64).copy()
    if out.ndim == 1:
        out = out[None, :]

    moon_idx = _moon_bs_indices_from_names(coef_names_local)
    if moon_idx.size < 3:
        return out.astype(np.float32)

    near = np.asarray(coef_near, dtype=np.float64)
    far = np.asarray(coef_far, dtype=np.float64)
    if near.ndim == 1:
        near = near[None, :]
    if far.ndim == 1:
        far = far[None, :]

    lam_grid_all = [float(v) for v in prior_cfg.get("lambda_grid", [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0])]
    min_lambda = float(prior_cfg.get("min_lambda", 0.03))
    lam_grid = [lam for lam in lam_grid_all if lam >= min_lambda]
    if len(lam_grid) == 0:
        lam_grid = [min_lambda]

    eps = float(prior_cfg.get("eps", 1e-12))
    target_scale = float(prior_cfg.get("target_roughness_scale", 0.90))
    d2 = _build_d2_matrix(moon_idx.size)

    for i in range(out.shape[0]):
        pred_row = out[i, moon_idx]
        near_row = near[i, moon_idx]
        far_row = far[i, moon_idx]

        target_rough = np.nanmedian([_row_spline_roughness(near_row), _row_spline_roughness(far_row)])
        if not np.isfinite(target_rough) or target_rough <= 0:
            target_rough = _row_spline_roughness(pred_row)
        target_rough = max(float(target_rough) * max(target_scale, 1e-6), eps)

        best_vals = np.clip(pred_row, 0.0, None)
        best_score = np.inf
        for lam in lam_grid:
            vals = _smooth_nonneg_with_lambda(pred_row, lam, d2=d2)
            rough = max(_row_spline_roughness(vals), eps)
            score = abs(np.log(rough) - np.log(target_rough))
            if score < best_score:
                best_score = score
                best_vals = vals

        out[i, moon_idx] = best_vals

    return out.astype(np.float32)


def predict_sci_coefficients_group_mlp_residual_corrected(artifacts_local, coef_near_phys, coef_far_phys, ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Predict coefficients with base deep-group MLP + learned ridge residual correction."""
    base = predict_sci_coefficients_group_mlp(
        artifacts_local,
        coef_near_phys=coef_near_phys,
        coef_far_phys=coef_far_phys,
        ctx_near_phys=ctx_near_phys,
        ctx_far_phys=ctx_far_phys,
        ctx_sci_phys=ctx_sci_phys,
    ).astype(np.float32)

    cn = np.asarray(coef_near_phys, dtype=np.float32)
    cf = np.asarray(coef_far_phys, dtype=np.float32)
    ctn = np.asarray(ctx_near_phys, dtype=np.float32)
    ctf = np.asarray(ctx_far_phys, dtype=np.float32)
    cts = np.asarray(ctx_sci_phys, dtype=np.float32)

    if cn.ndim == 1:
        cn = cn[None, :]
    if cf.ndim == 1:
        cf = cf[None, :]
    if ctn.ndim == 1:
        ctn = ctn[None, :]
    if ctf.ndim == 1:
        ctf = ctf[None, :]
    if cts.ndim == 1:
        cts = cts[None, :]

    dctx = cts - 0.5 * (ctn + ctf)
    X = np.hstack([base, cn, cf, dctx, cts]).astype(np.float32)
    corrected = base + coef_residual_corrector.predict(X).astype(np.float32)

    prior_cfg = artifacts_local.get("moon_spline_prior", MOON_SPLINE_PRIOR_CONFIG)
    if bool(prior_cfg.get("enabled", True)):
        corrected = _apply_moon_spline_prior_batch(
            corrected,
            cn,
            cf,
            coef_names_local=artifacts_local.get("coef_names", []),
            prior_cfg=prior_cfg,
        )

    return np.clip(corrected, 0.0, None)


# Keep prior settings with artifacts so behavior is explicit and reproducible.
cvae_artifacts["moon_spline_prior"] = dict(MOON_SPLINE_PRIOR_CONFIG)
print("Moon spline prior config:", cvae_artifacts["moon_spline_prior"])


# Compatibility alias used by some existing cells.
def predict_sci_coefficients_cvae_residual_corrected(cvae_artifacts_local, coef_near_phys, coef_far_phys, ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Compatibility alias: residual-corrected deep-group predictor."""
    return predict_sci_coefficients_group_mlp_residual_corrected(
        cvae_artifacts_local,
        coef_near_phys=coef_near_phys,
        coef_far_phys=coef_far_phys,
        ctx_near_phys=ctx_near_phys,
        ctx_far_phys=ctx_far_phys,
        ctx_sci_phys=ctx_sci_phys,
    )


def predict_sci_coefficients_default(cvae_artifacts_local, coef_near_phys, coef_far_phys, ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Default notebook inference path (corrected coefficients + Moon spline smoothness prior)."""
    return predict_sci_coefficients_group_mlp_residual_corrected(
        cvae_artifacts_local,
        coef_near_phys=coef_near_phys,
        coef_far_phys=coef_far_phys,
        ctx_near_phys=ctx_near_phys,
        ctx_far_phys=ctx_far_phys,
        ctx_sci_phys=ctx_sci_phys,
    )

In [ ]:
# Optional quick sanity print for this notebook variant
print("Deep group-head MLP + residual notebook variant is configured.")
print("Use predict_sci_coefficients_default(...) in downstream diagnostics and reconstruction cells.")

## Coefficient Model Diagnostics

These diagnostics summarize grouped coefficient behavior versus context for near/far/true-science/predicted-science coefficient summaries.

In [ ]:
# Relationship plots: grouped coefficient-vs-context structure using default predictor
import pandas as pd
import plotly.express as px

required = ["cvae_artifacts", "filtered_triplet", "predict_sci_coefficients_default"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training + residual-correction cells first. Missing: " + ", ".join(missing))

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float32)
ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)
coef_names_all = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names_all = [str(n) for n in filtered_triplet["ctx_names"]]

coef_pred_all = predict_sci_coefficients_default(
    cvae_artifacts,
    coef_near_phys=coef_near_all,
    coef_far_phys=coef_far_all,
    ctx_near_phys=ctx_near_all,
    ctx_far_phys=ctx_far_all,
    ctx_sci_phys=ctx_sci_all,
)

coef_name_l = [n.lower() for n in coef_names_all]

def _group_idx(names_l):
    groups = {
        "diffuse_continuum_median": [
            i for i, n in enumerate(names_l)
            if (n.startswith("ho2") or n.startswith("feo") or n.startswith("o2") or "diffuse" in n or "continuum" in n)
        ],
        "oh_median": [i for i, n in enumerate(names_l) if n.startswith("oh")],
        "atomic_median": [i for i, n in enumerate(names_l) if n.startswith("atom")],
    }
    return {k: np.asarray(v, dtype=int) for k, v in groups.items() if len(v) > 0}

group_idx = _group_idx(coef_name_l)
if len(group_idx) == 0:
    raise RuntimeError("No coefficient groups found for relationship diagnostics")

rows_rel = []
max_points = 6000
n_rows = coef_sci_all.shape[0]
if n_rows > max_points:
    rng = np.random.default_rng(42)
    use = np.sort(rng.choice(n_rows, size=max_points, replace=False))
else:
    use = np.arange(n_rows)

for gname, idx in group_idx.items():
    near_g = np.nanmedian(coef_near_all[use][:, idx], axis=1)
    far_g = np.nanmedian(coef_far_all[use][:, idx], axis=1)
    sci_true_g = np.nanmedian(coef_sci_all[use][:, idx], axis=1)
    sci_pred_g = np.nanmedian(coef_pred_all[use][:, idx], axis=1)

    for j, cname in enumerate(ctx_names_all):
        x = ctx_sci_all[use, j]
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": near_g, "series": "near"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": far_g, "series": "far"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": sci_true_g, "series": "sci_true"}))
        rows_rel.append(pd.DataFrame({"group": gname, "context_param": cname, "context_value": x, "coef_value": sci_pred_g, "series": "sci_pred_default"}))

rel_df = pd.concat(rows_rel, ignore_index=True)

fig_rel = px.scatter(
    rel_df,
    x="context_value",
    y="coef_value",
    color="series",
    facet_col="context_param",
    facet_row="group",
    opacity=0.20,
    render_mode="webgl",
    title="Grouped coefficient relationships vs context (near/far/true/default)",
    color_discrete_map={
        "near": "#1f77b4",
        "far": "#9467bd",
        "sci_true": "#2ca02c",
        "sci_pred_default": "#d62728",
    },
)
fig_rel.for_each_annotation(
    lambda a: a.update(text=a.text.replace("group=", "").replace("context_param=", ""))
)
fig_rel.update_xaxes(matches=None)
fig_rel.update_yaxes(matches=None)
fig_rel.update_layout(template="plotly_white", height=max(750, 210 * len(group_idx)))
fig_rel.show()

In [ ]:
# Deep-group diagnostics split into two views:
# 1) trunk-embedding vs context grid
# 2) targeted coefficient predictions vs context (ND-style groups)
import numpy as np
import pandas as pd
import plotly.express as px
import torch

required = ["cvae_artifacts", "filtered_triplet", "predict_sci_coefficients_default"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training/model cells first. Missing: " + ", ".join(missing))

model = cvae_artifacts["model"]
device = cvae_artifacts["device"]
coef_scaler = cvae_artifacts["coef_scaler"]
ctx_scaler = cvae_artifacts["ctx_scaler"]

test_idx = np.asarray(cvae_artifacts["test_idx"], dtype=int)

coef_near = np.asarray(filtered_triplet["coef_near"][test_idx], dtype=np.float32)
coef_far = np.asarray(filtered_triplet["coef_far"][test_idx], dtype=np.float32)
coef_sci = np.asarray(filtered_triplet["coef_sci"][test_idx], dtype=np.float32)
ctx_near = np.asarray(filtered_triplet["ctx_near"][test_idx], dtype=np.float32)
ctx_far = np.asarray(filtered_triplet["ctx_far"][test_idx], dtype=np.float32)
ctx_sci = np.asarray(filtered_triplet["ctx_sci"][test_idx], dtype=np.float32)

coef_names = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names = [str(n) for n in filtered_triplet["ctx_names"]]

near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near)), -25.0, 25.0).astype(np.float32)
far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far)), -25.0, 25.0).astype(np.float32)
near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)
delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

X_n = np.hstack([near_n, far_n, delta_ctx_n, sci_ctx_n]).astype(np.float32)
with torch.no_grad():
    xb = torch.from_numpy(X_n).to(device)
    emb_t = model.trunk(xb)

# Use the notebook default predictor (deep-group MLP + residual correction) for output diagnostics.
coef_pred_default = predict_sci_coefficients_default(
    cvae_artifacts,
    coef_near_phys=coef_near,
    coef_far_phys=coef_far,
    ctx_near_phys=ctx_near,
    ctx_far_phys=ctx_far,
    ctx_sci_phys=ctx_sci,
)


def _robust_bounds(arr, q_low=1.0, q_high=99.0, pad_frac=0.05):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (-1.0, 1.0)

    lo, hi = np.nanpercentile(arr, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = np.nanmin(arr)
        hi = np.nanmax(arr)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        c = float(arr[0])
        lo, hi = c - 0.5, c + 0.5

    span = max(hi - lo, 1e-6)
    pad = pad_frac * span
    return lo - pad, hi + pad


def _clip_to_bounds(arr, bounds):
    return np.clip(np.asarray(arr, dtype=np.float64), bounds[0], bounds[1])


def _set_bottom_facet_x_titles(fig, col_names, n_rows):
    n_cols = len(col_names)
    for r in range(n_rows):
        for c, cname in enumerate(col_names):
            axis_idx = r * n_cols + c + 1
            axis_name = "xaxis" if axis_idx == 1 else f"xaxis{axis_idx}"
            if axis_name in fig.layout:
                fig.layout[axis_name].title.text = cname if r == 0 else ""


def _set_left_facet_y_titles(fig, row_titles, n_cols):
    ordered = list(row_titles)[::-1]
    for r, title in enumerate(ordered):
        axis_idx = r * n_cols + 1
        axis_name = "yaxis" if axis_idx == 1 else f"yaxis{axis_idx}"
        if axis_name in fig.layout:
            fig.layout[axis_name].title.text = title


def _is_time_context(name):
    lname = str(name).strip().lower()
    exact = {
        "obstime_mjd",
        "obstime_mjd_z",
        "obstime_year_sin",
        "obstime_year_cos",
        "obstime_day_sin",
        "obstime_day_cos",
    }
    return lname in exact or lname.startswith("obstime")


# ----- Shared context ordering + split into non-time vs time -----
ctx_names_l = [str(c).strip().lower() for c in ctx_names]
preferred_ctx = ["alt", "moon_sep", "moon_alt", "sun_alt", "moon_illum", "airmass"]
ctx_order = [c for c in preferred_ctx if c in ctx_names_l]
ctx_order += [c for c in ctx_names_l if c not in ctx_order]
ctx_sel = [ctx_names_l.index(c) for c in ctx_order]
ctx_labels = [ctx_names[i] for i in ctx_sel]

ctx_sel_non_time = [idx for idx in ctx_sel if not _is_time_context(ctx_names[idx])]
ctx_labels_non_time = [ctx_names[idx] for idx in ctx_sel_non_time]
ctx_sel_time = [idx for idx in ctx_sel if _is_time_context(ctx_names[idx])]
ctx_labels_time = [ctx_names[idx] for idx in ctx_sel_time]

print(f"Context split: non-time={len(ctx_labels_non_time)} | time={len(ctx_labels_time)}")
if "airmass" in ctx_names_l:
    print("Embedding-context plot includes airmass.")
else:
    print("Warning: airmass not found in context columns; embedding-context plot cannot include it.")

# ----- Plot 1: trunk embedding vs context -----
emb_np = emb_t.cpu().numpy().astype(np.float64)
emb_var = np.nanvar(emb_np, axis=0)
max_embed_dims = int(min(6, emb_np.shape[1]))
embed_sel = np.argsort(emb_var)[-max_embed_dims:][::-1]
embed_labels = [f"h{int(i)}" for i in embed_sel]

max_points = 5000
n_rows = emb_np.shape[0]
if n_rows > max_points:
    rng = np.random.default_rng(42)
    use = np.sort(rng.choice(n_rows, size=max_points, replace=False))
else:
    use = np.arange(n_rows)


def _plot_embed_vs_context(ctx_labels_use, ctx_sel_use, title):
    if len(ctx_sel_use) == 0:
        print(f"Skipping {title}: no matching context dimensions.")
        return

    rows_embed = []
    for k_plot, k_emb in enumerate(embed_sel):
        h = emb_np[use, k_emb]
        h_bounds = _robust_bounds(h)
        for j_plot, j_ctx in enumerate(ctx_sel_use):
            x = ctx_sci[use, j_ctx]
            x_bounds = _robust_bounds(x)
            rows_embed.append(
                pd.DataFrame(
                    {
                        "embedding_dim": embed_labels[k_plot],
                        "context_param": ctx_labels_use[j_plot],
                        "context_value": _clip_to_bounds(x, x_bounds),
                        "embedding_value": _clip_to_bounds(h, h_bounds),
                    }
                )
            )

    embed_ctx_df = pd.concat(rows_embed, ignore_index=True)

    fig_embed_ctx = px.scatter(
        embed_ctx_df,
        x="context_value",
        y="embedding_value",
        facet_col="context_param",
        facet_row="embedding_dim",
        opacity=0.24,
        render_mode="webgl",
        title=title,
        labels={"embedding_value": "embedding value", "context_value": ""},
    )
    fig_embed_ctx.for_each_annotation(
        lambda a: a.update(text=a.text.replace("embedding_dim=", "").replace("context_param=", ""))
    )
    fig_embed_ctx.update_xaxes(matches=None)
    fig_embed_ctx.update_yaxes(matches=None)
    fig_embed_ctx.update_layout(template="plotly_white", height=max(780, 220 * max_embed_dims))
    _set_bottom_facet_x_titles(fig_embed_ctx, ctx_labels_use, max_embed_dims)
    fig_embed_ctx.show()


_plot_embed_vs_context(
    ctx_labels_non_time,
    ctx_sel_non_time,
    "Deep-group trunk embedding vs non-time context grid",
)
_plot_embed_vs_context(
    ctx_labels_time,
    ctx_sel_time,
    "Deep-group trunk embedding vs time-context grid",
)


# ----- Plot 2: ND-style targeted coefficient predictions vs context -----
coef_name_l = [str(n).lower() for n in coef_names]

moon_bs_idx = np.array(
    [i for i, n in enumerate(coef_name_l) if n.startswith("moon_bs")],
    dtype=int,
)


def _continuum_group(name):
    # Explicitly skip OH-related groups/components in this pass.
    if "oh" in name:
        return None

    patterns = [
        ("moon", ["moon", "zodi"]),
        ("diffuse", ["diffuse"]),
        ("ho2", ["ho2", "hydroperoxyl"]),
        ("feo", ["feo", "iron"]),
        ("o2", ["o2", "oxygen"]),
        ("continuum", ["continuum"]),
    ]
    for gname, keys in patterns:
        if any(k in name for k in keys):
            return gname
    return None


comp_to_idx = {}
for i, n in enumerate(coef_name_l):
    if i in moon_bs_idx:
        continue
    g = _continuum_group(n)
    if g is None:
        continue
    comp_to_idx.setdefault(g, []).append(i)

row_groups = []
if moon_bs_idx.size > 0:
    row_groups.append(("moon_bs_median", moon_bs_idx))


def _find_prefixed_index(names, prefix, target_id):
    prefix = str(prefix).lower()
    target_num = int(target_id)
    for i, n in enumerate(names):
        lname = str(n).lower()
        if not lname.startswith(prefix):
            continue
        tail = "".join(ch for ch in lname[len(prefix):] if ch.isdigit())
        if tail and int(tail) == target_num:
            return i
    return None


for tid in [4, 12, 20]:
    idx = _find_prefixed_index(coef_names, "moon_bs", tid)
    if idx is None:
        print(f"moon_bs{tid} not found in coef_names; skipping in targeted plot.")
        continue
    row_groups.append((f"moon_bs{int(tid):02d}", np.array([idx], dtype=int)))

for tid in [100, 200, 300]:
    idx = _find_prefixed_index(coef_names, "oh", tid)
    if idx is None:
        print(f"OH{tid} not found in coef_names; skipping in targeted plot.")
        continue
    row_groups.append((f"oh{int(tid):02d}", np.array([idx], dtype=int)))

for gname in sorted(comp_to_idx.keys()):
    row_groups.append((gname, np.array(comp_to_idx[gname], dtype=int)))

if len(row_groups) == 0:
    raise RuntimeError(
        "Could not identify ND-style Moon_bs/OH/continuum groups from coefficient names. "
        "Inspect coef_names naming conventions."
    )


def _build_target_df(ctx_labels_use, ctx_sel_use):
    rows_out = []
    for row_name, idxs in row_groups:
        y_true = np.nanmedian(coef_sci[:, idxs], axis=1)
        y_pred = np.nanmedian(coef_pred_default[:, idxs], axis=1)

        for j, cname in enumerate(ctx_labels_use):
            j_ctx = ctx_sel_use[j]
            x_raw = ctx_sci[:, j_ctx]
            x_bounds = _robust_bounds(x_raw)
            y_bounds = _robust_bounds(np.concatenate([y_true, y_pred]))

            rows_out.append(
                pd.DataFrame(
                    {
                        "output_group": row_name,
                        "context_param": cname,
                        "context_value": _clip_to_bounds(x_raw, x_bounds),
                        "output_value": _clip_to_bounds(y_true, y_bounds),
                        "series": "true",
                    }
                )
            )
            rows_out.append(
                pd.DataFrame(
                    {
                        "output_group": row_name,
                        "context_param": cname,
                        "context_value": _clip_to_bounds(x_raw, x_bounds),
                        "output_value": _clip_to_bounds(y_pred, y_bounds),
                        "series": "default_pred",
                    }
                )
            )

    if len(rows_out) == 0:
        return None
    return pd.concat(rows_out, ignore_index=True)


def _plot_target_coef_vs_context(out_df, ctx_labels_use, title):
    if out_df is None or len(ctx_labels_use) == 0:
        print(f"Skipping {title}: no matching context dimensions.")
        return

    fig_coef_ctx = px.scatter(
        out_df,
        x="context_value",
        y="output_value",
        color="series",
        facet_col="context_param",
        facet_row="output_group",
        opacity=0.28,
        render_mode="webgl",
        title=title,
        color_discrete_map={"true": "#1f77b4", "default_pred": "#d62728"},
        labels={"output_value": "", "context_value": ""},
    )
    fig_coef_ctx.for_each_annotation(
        lambda a: a.update(text=a.text.replace("context_param=", "").replace("output_group=", ""))
    )
    fig_coef_ctx.update_xaxes(matches=None)
    fig_coef_ctx.update_yaxes(matches=None)
    fig_coef_ctx.update_layout(template="plotly_white", height=max(900, 220 * len(row_groups)))
    _set_bottom_facet_x_titles(fig_coef_ctx, ctx_labels_use, len(row_groups))
    _set_left_facet_y_titles(fig_coef_ctx, [name for name, _ in row_groups], len(ctx_labels_use))
    fig_coef_ctx.show()


out_df_non_time = _build_target_df(ctx_labels_non_time, ctx_sel_non_time)
out_df_time = _build_target_df(ctx_labels_time, ctx_sel_time)

_plot_target_coef_vs_context(
    out_df_non_time,
    ctx_labels_non_time,
    "ND-style targeted coefficient summaries vs non-time context (true vs default corrected)",
)
_plot_target_coef_vs_context(
    out_df_time,
    ctx_labels_time,
    "ND-style targeted coefficient summaries vs time context (true vs default corrected)",
)

## Full-Spectrum Test From Predicted Coefficients

> Primary workflow: predict SCI decomposition coefficients, then reconstruct SCI spectrum with the decomposition forward model.

Data usage policy:
- `*_meta_coef.fits` products are used only for **training** the coefficient model.
- The four `*_every10*.fits` files are used only for **testing** and are never seen during training.

This cell reconstructs only the requested row from the every10 set:
1. Predict science coefficients for that row.
2. Reconstruct the science spectrum using the row wavelength grid and LSF.
3. Plot reconstructed vs observed spectra and relative residuals.


In [ ]:
# Full-spectrum reconstruction test for a single requested row using default coefficients
import plotly.graph_objects as go

EVERY10_INPUT = "lvmsframe_median_stack_1.2.1_every10.fits"
EVERY10_NEAR = "lvmsframe_median_stack_1.2.1_every10_decomp_sky1.fits"
EVERY10_FAR = "lvmsframe_median_stack_1.2.1_every10_decomp_sky2.fits"
EVERY10_SCI = "lvmsframe_median_stack_1.2.1_every10_decomp_sci.fits"

# Set the row to reconstruct and inspect.
REQUESTED_ROW = 900

required = [
    "cvae_artifacts",
    "predict_sci_coefficients_default",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "_infer_base_dir_for_reconstruction",
    "_moon_bs_indices_from_names",
    "_row_spline_roughness",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the training + residual-correction cells first. Missing: " + ", ".join(missing))

# 1) Load coefficients/context from every10 decomposition products.
e10_triplet = build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT,
    sky_near_decomp_fits_path=EVERY10_NEAR,
    sky_far_decomp_fits_path=EVERY10_FAR,
    sci_decomp_fits_path=EVERY10_SCI,
    context_columns=context_cols,
    return_chi2=False,
)
n_e10 = int(e10_triplet["n_rows"])
coef_names_e10 = [str(n) for n in e10_triplet["coef_names"]]
moon_idx = _moon_bs_indices_from_names(coef_names_e10)

if moon_idx.size < 3:
    raise RuntimeError("Expected at least 3 Moon_bs coefficients for spline diagnostics")

# 2) Load observed spectra, wavelength grid, and LSF from every10 input.
with fits.open(EVERY10_INPUT) as hdul:
    for ext in ("FLUX_SKY_NEAR", "FLUX_SKY_FAR", "FLUX_SCI", "WAVE", "LSF_SCI"):
        if ext not in hdul:
            raise KeyError(f"Missing extension {ext} in {EVERY10_INPUT}")

    wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
    flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
    flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
    flux_sci_true_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
    lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)

n_spec, n_wave = flux_sci_true_all.shape
if n_e10 != n_spec:
    raise ValueError(
        f"Row count mismatch: triplet has {n_e10} rows, spectra have {n_spec} rows. "
        "Check that every10 decomposition files were produced from the same input file."
    )

idx_row = int(REQUESTED_ROW)
if idx_row < 0 or idx_row >= n_spec:
    raise IndexError(f"REQUESTED_ROW={idx_row} out of range [0, {n_spec - 1}]")

# Normalize WAVE/LSF arrays to per-row vectors, then select requested row.
wave_row = wave_arr if wave_arr.ndim == 1 else wave_arr[idx_row]
lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else lsf_sci_arr[idx_row]

flux_near_row = flux_near_all[idx_row]
flux_far_row = flux_far_all[idx_row]
flux_sci_true_row = flux_sci_true_all[idx_row]

# 3) Predict SCI coefficients for the requested row using global default path.
coef_pred_row_batch = predict_sci_coefficients_default(
    cvae_artifacts,
    coef_near_phys=e10_triplet["coef_near"][idx_row: idx_row + 1],
    coef_far_phys=e10_triplet["coef_far"][idx_row: idx_row + 1],
    ctx_near_phys=e10_triplet["ctx_near"][idx_row: idx_row + 1],
    ctx_far_phys=e10_triplet["ctx_far"][idx_row: idx_row + 1],
    ctx_sci_phys=e10_triplet["ctx_sci"][idx_row: idx_row + 1],
)
coef_pred_row = np.asarray(coef_pred_row_batch[0], dtype=np.float64)

# 3b) Moon_bs coefficient diagnostics (global prior already applied).
coef_near_row = np.asarray(e10_triplet["coef_near"][idx_row], dtype=np.float64)
coef_far_row = np.asarray(e10_triplet["coef_far"][idx_row], dtype=np.float64)
coef_sci_row = np.asarray(e10_triplet["coef_sci"][idx_row], dtype=np.float64)
moon_pred = coef_pred_row[moon_idx]
moon_near = coef_near_row[moon_idx]
moon_far = coef_far_row[moon_idx]
moon_true = coef_sci_row[moon_idx]

# 4) Reconstruct this row for SCI prediction and near/far self-consistency checks.
base_dir_guess = _infer_base_dir_for_reconstruction()
comps_sci = reconstruct_component_spectra(
    wave=wave_row,
    coef=coef_pred_row,
    lsf_sigma=lsf_row / 2.35,
    n_spline_knots=25,
    base_dir=base_dir_guess,
)
comps_near_from_near = reconstruct_component_spectra(
    wave=wave_row,
    coef=coef_near_row,
    lsf_sigma=lsf_row / 2.35,
    n_spline_knots=25,
    base_dir=base_dir_guess,
)
comps_far_from_far = reconstruct_component_spectra(
    wave=wave_row,
    coef=coef_far_row,
    lsf_sigma=lsf_row / 2.35,
    n_spline_knots=25,
    base_dir=base_dir_guess,
)

flux_sci_pred_row = np.asarray(comps_sci["total"], dtype=np.float64) / FACTOR
flux_near_recon_row = np.asarray(comps_near_from_near["total"], dtype=np.float64) / FACTOR
flux_far_recon_row = np.asarray(comps_far_from_far["total"], dtype=np.float64) / FACTOR

# 5) Single-row metrics.
resid_row = flux_sci_pred_row - flux_sci_true_row
rmse_row = float(np.sqrt(np.mean(resid_row ** 2)))
mae_row = float(np.mean(np.abs(resid_row)))
rel_resid_row = resid_row / np.where(flux_sci_true_row != 0, flux_sci_true_row, np.nan)

rmse_near_recon = float(np.sqrt(np.mean((flux_near_recon_row - flux_near_row) ** 2)))
rmse_far_recon = float(np.sqrt(np.mean((flux_far_recon_row - flux_far_row) ** 2)))

prior_cfg = cvae_artifacts.get("moon_spline_prior", {})
print("Single-row reconstruction summary (every10, default coefficients)")
print(f"  row index  = {idx_row}")
print(f"  n_wave     = {n_wave}")
print("  predictor  = deep group-head MLP + ridge residual correction")
print(f"  global Moon spline prior enabled = {prior_cfg.get('enabled', None)}")
print(f"  global Moon spline lambda grid   = {prior_cfg.get('lambda_grid', None)}")
print(
    "  Moon_bs roughness: "
    f"pred={_row_spline_roughness(moon_pred):.4g}, "
    f"near={_row_spline_roughness(moon_near):.4g}, "
    f"far={_row_spline_roughness(moon_far):.4g}, "
    f"sci_true={_row_spline_roughness(moon_true):.4g}"
)
print(f"  near self-recon RMSE = {rmse_near_recon:.6g}")
print(f"  far self-recon RMSE  = {rmse_far_recon:.6g}")
print(f"  sci row RMSE         = {rmse_row:.6g}")
print(f"  sci row MAE          = {mae_row:.6g}")

# 6) Four-panel diagnostic plot:
#    row1: near observed vs reconstruction from near coefficients
#    row2: far observed vs reconstruction from far coefficients
#    row3: science true vs science prediction
#    row4: science relative residual
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "Near: observed vs reconstructed from near coefficients",
        "Far: observed vs reconstructed from far coefficients",
        "Science: true vs predicted",
        "Science residual: (pred - true) / true",
    ),
    row_heights=[0.24, 0.24, 0.34, 0.18],
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_row * FACTOR,
        mode="lines",
        name="near true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_recon_row * FACTOR,
        mode="lines",
        name="near recon(from near coef)",
        line=dict(color="#e41a1c", width=1.4),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_row * FACTOR,
        mode="lines",
        name="far true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_recon_row * FACTOR,
        mode="lines",
        name="far recon(from far coef)",
        line=dict(color="#ff7f00", width=1.4),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_true_row * FACTOR,
        mode="lines",
        name="science true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_pred_row * FACTOR,
        mode="lines",
        name="science recon(pred)",
        line=dict(color="#1f78b4", width=1.4),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=rel_resid_row,
        mode="lines",
        name="science residual",
        line=dict(color="#d62728", width=1.0),
    ),
    row=4,
    col=1,
)
fig.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=4, col=1)

fig.update_yaxes(type="log", title_text="Near flux", row=1, col=1)
fig.update_yaxes(type="log", title_text="Far flux", row=2, col=1)
fig.update_yaxes(type="log", title_text="Science flux", row=3, col=1)
fig.update_yaxes(type="linear", title_text="(pred-true)/true", row=4, col=1)
fig.update_xaxes(title_text="Wavelength [A]", row=4, col=1)

fig.update_layout(
    template="plotly_white",
    title=(
        f"Every10 row {idx_row} | near RMSE={rmse_near_recon:.3g}, "
        f"far RMSE={rmse_far_recon:.3g}, sci RMSE={rmse_row:.3g}"
    ),
    height=1160,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.show()

# 7) Moon spline coefficient diagnostic figure (global-prior result).
moon_axis = np.arange(moon_idx.size)
fig_moon = go.Figure()
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_near,
        mode="lines+markers",
        name="near",
        line=dict(color="#7f7f7f"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_far,
        mode="lines+markers",
        name="far",
        line=dict(color="#bdbdbd"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_true,
        mode="lines+markers",
        name="sci true",
        line=dict(color="#1f78b4"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_pred,
        mode="lines+markers",
        name="pred default",
        line=dict(color="#e41a1c"),
    )
)
fig_moon.update_layout(
    template="plotly_white",
    title="Moon spline coefficients (global-prior prediction) for selected row",
    xaxis_title="Moon_bs coefficient index",
    yaxis_title="coefficient value",
    height=420,
)
fig_moon.show()

In [ ]:
# Embedding outlier diagnostics: raw trunk embeddings vs train-quantile soft-capped embeddings
import numpy as np
import torch
import torch.nn.functional as F

required = ["cvae_artifacts", "filtered_triplet", "_coef_to_model_space"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training/model cells first. Missing: " + ", ".join(missing))

model = cvae_artifacts["model"]
device = cvae_artifacts["device"]
coef_scaler = cvae_artifacts["coef_scaler"]
ctx_scaler = cvae_artifacts["ctx_scaler"]
train_idx = np.asarray(cvae_artifacts["train_idx"], dtype=int)
test_idx = np.asarray(cvae_artifacts["test_idx"], dtype=int)

coef_near = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
ctx_near = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)

near_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_near)), -25.0, 25.0).astype(np.float32)
far_n = np.clip(coef_scaler.transform(_coef_to_model_space(coef_far)), -25.0, 25.0).astype(np.float32)
near_ctx_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
far_ctx_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
sci_ctx_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)
delta_ctx_n = sci_ctx_n - 0.5 * (near_ctx_n + far_ctx_n)

X_all = np.hstack([near_n, far_n, delta_ctx_n, sci_ctx_n]).astype(np.float32)

with torch.no_grad():
    emb_tr = model.trunk(torch.from_numpy(X_all[train_idx]).to(device))
    emb_te = model.trunk(torch.from_numpy(X_all[test_idx]).to(device))

emb_tr_np = emb_tr.detach().cpu().numpy().astype(np.float64)
emb_lo = np.nanpercentile(emb_tr_np, 0.1, axis=0).astype(np.float32)
emb_hi = np.nanpercentile(emb_tr_np, 99.9, axis=0).astype(np.float32)

with torch.no_grad():
    emb_te_t = emb_te
    emb_lo_t = torch.from_numpy(emb_lo).to(device)
    emb_hi_t = torch.from_numpy(emb_hi).to(device)
    emb_cap_w = 0.05 * torch.clamp(emb_hi_t - emb_lo_t, min=1e-6)
    emb_excess_hi = F.softplus((emb_te_t - emb_hi_t) / emb_cap_w) * emb_cap_w
    emb_excess_lo = F.softplus((emb_lo_t - emb_te_t) / emb_cap_w) * emb_cap_w
    emb_te_clip = emb_te_t - emb_excess_hi + emb_excess_lo

raw_exceed = float(((emb_te_t > emb_hi_t) | (emb_te_t < emb_lo_t)).float().mean().item())
soft_adjust = float((torch.abs(emb_te_clip - emb_te_t) > 1e-7).float().mean().item())

emb_raw_np = emb_te_t.detach().cpu().numpy().astype(np.float64)
emb_clip_np = emb_te_clip.detach().cpu().numpy().astype(np.float64)


def _embedding_cleanliness(x):
    if x.ndim != 2 or x.shape[1] < 2:
        return np.nan
    x = x - np.nanmean(x, axis=0, keepdims=True)
    std = np.nanstd(x, axis=0, keepdims=True)
    x = x / np.maximum(std, 1e-8)
    c = np.corrcoef(x, rowvar=False)
    offdiag = c[~np.eye(c.shape[0], dtype=bool)]
    return float(1.0 - np.nanmean(np.abs(offdiag)))

clean_raw = _embedding_cleanliness(emb_raw_np)
clean_clip = _embedding_cleanliness(emb_clip_np)

print(f"Embedding exceedance vs train q[0.1,99.9]: {100.0 * raw_exceed:.2f}%")
print(f"Embedding soft-cap adjustment fraction: {100.0 * soft_adjust:.2f}%")
print(f"Embedding cleanliness raw trunk features: {clean_raw:.4f}")
print(f"Embedding cleanliness soft-capped features: {clean_clip:.4f}")

In [ ]:
# Additional embedding-structure diagnostics
import numpy as np


def _embedding_stats(x, name):
    x = np.asarray(x, dtype=np.float64)
    var = np.var(x, axis=0)
    pr = float((np.sum(var) ** 2) / max(np.sum(var ** 2), 1e-12))

    z = x - np.mean(x, axis=0, keepdims=True)
    z = z / np.maximum(np.std(z, axis=0, keepdims=True), 1e-8)
    c = np.corrcoef(z, rowvar=False)
    offdiag = c[~np.eye(c.shape[0], dtype=bool)]

    print(f"{name} participation ratio: {pr:.3f} / {x.shape[1]} dims")
    print(f"{name} mean |offdiag corr|: {np.mean(np.abs(offdiag)):.4f}")
    print(f"{name} max |offdiag corr| : {np.max(np.abs(offdiag)):.4f}")


_embedding_stats(emb_tr_np, "train trunk embedding")
_embedding_stats(emb_raw_np, "test trunk embedding")
_embedding_stats(emb_clip_np, "test trunk embedding soft-capped")

In [ ]:
# Training schedule audit: recommend total epochs and patience from deep-group history
import numpy as np
import pandas as pd
import plotly.express as px

required = ["cvae_artifacts", "deep_group_config"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training cells first. Missing: " + ", ".join(missing))

hist = pd.DataFrame(cvae_artifacts["history"]).copy()
if hist.empty:
    raise RuntimeError("cvae_artifacts['history'] is empty")

best_epoch = int(cvae_artifacts["best_epoch"])
train_epochs = int(deep_group_config.get("n_epochs", int(hist["epoch"].max())))
patience = int(deep_group_config.get("patience", 8))

# First epoch whose val loss is within a tolerance band of the global best.
best_val = float(hist["val_loss"].min())
within_1pct = hist.loc[hist["val_loss"] <= best_val * 1.01, "epoch"]
within_2pct = hist.loc[hist["val_loss"] <= best_val * 1.02, "epoch"]
first_1pct = int(within_1pct.iloc[0]) if len(within_1pct) else best_epoch
first_2pct = int(within_2pct.iloc[0]) if len(within_2pct) else best_epoch

# Plateau estimate using trailing window improvements in val loss.
win = 8
plateau_epoch = best_epoch
if len(hist) >= 2 * win:
    vals = hist["val_loss"].to_numpy(dtype=float)
    epochs = hist["epoch"].to_numpy(dtype=int)
    for i in range(win, len(vals) - win):
        prev_min = np.min(vals[max(0, i - win):i])
        next_min = np.min(vals[i:i + win])
        if prev_min <= 0:
            continue
        rel_gain = (prev_min - next_min) / prev_min
        if rel_gain < 0.002:
            plateau_epoch = int(epochs[i])
            break

# Heuristic recommendation.
rec_epochs = int(min(max(train_epochs, 20), max(first_2pct + 6, plateau_epoch + 4, best_epoch + 4)))
rec_patience = int(np.clip(round(0.20 * rec_epochs), 5, 12))

summary = pd.DataFrame([
    {
        "configured_epochs": train_epochs,
        "configured_patience": patience,
        "best_epoch": best_epoch,
        "first_epoch_within_1pct_best": first_1pct,
        "first_epoch_within_2pct_best": first_2pct,
        "plateau_epoch_estimate": plateau_epoch,
        "recommended_epochs": rec_epochs,
        "recommended_patience": rec_patience,
    }
])

print("Deep-group training schedule summary")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

hist_long = hist[["epoch", "train_loss", "val_loss"]].copy()
fig = px.line(
    hist_long,
    x="epoch",
    y=["train_loss", "val_loss"],
    title="Deep-group MLP training curves",
)
fig.add_vline(x=best_epoch, line_dash="dot", line_color="red")
fig.update_layout(template="plotly_white", height=500)
fig.show()

In [ ]:
# Mini sweep: training length / patience schedule (auto-consumes dataset-size suggestions)
import math
import numpy as np
import pandas as pd

required = ["filtered_triplet", "train_coeff_prediction_group_mlp", "predict_sci_coefficients_group_mlp"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run prerequisite cells first. Missing: " + ", ".join(missing))

if "cand_df" in globals() and isinstance(cand_df, pd.DataFrame) and "n_epochs" in cand_df.columns:
    sweep_source = cand_df[["n_epochs"]].copy().dropna().drop_duplicates().sort_values(["n_epochs"])
    source_label = "cand_df from dataset-size helper"
else:
    # Fallback: regenerate a compact candidate grid if helper cell was not run.
    batch_size_local = int(globals().get("BATCH_SIZE", 256))
    target_total_steps_local = int(globals().get("TARGET_TOTAL_STEPS", 360))

    n_rows_local = int(np.asarray(filtered_triplet["coef_sci"]).shape[0])
    steps_per_epoch_local = int(math.ceil(n_rows_local / float(batch_size_local)))
    base_epochs_local = int(max(20, round(target_total_steps_local / max(steps_per_epoch_local, 1))))

    candidates_local = []
    for ep in sorted(
        set([
            max(20, int(round(base_epochs_local * 0.75))),
            base_epochs_local,
            int(round(base_epochs_local * 1.25)),
        ])
    ):
        candidates_local.append({"n_epochs": ep})

    sweep_source = pd.DataFrame(candidates_local)
    source_label = "auto-fallback candidates"

sweep = []
for _, row in sweep_source.iterrows():
    ep = int(row["n_epochs"])
    pat = int(np.clip(round(0.20 * ep), 5, 12))
    sweep.append({"name": f"ep{ep}_pat{pat}", "n_epochs": ep, "patience": pat})

if len(sweep) == 0:
    raise RuntimeError("No schedule candidates found for sweep")

print(f"Using {len(sweep)} schedule candidates from: {source_label}")
print(pd.DataFrame(sweep)[["name", "n_epochs", "patience"]].to_string(index=False))

rows = []
for cfg in sweep:
    print("\n=== Schedule", cfg["name"], "===")
    art = train_coeff_prediction_group_mlp(
        filtered_triplet,
        n_epochs=int(cfg["n_epochs"]),
        batch_size=256,
        lr=1e-3,
        trunk_dims=(1024, 512, 256),
        head_dim=256,
        weight_decay=1e-4,
        grad_clip=1.0,
        patience=int(cfg["patience"]),
        seed=42,
    )

    test_idx = np.asarray(art["test_idx"], dtype=int)
    y_true = np.asarray(filtered_triplet["coef_sci"][test_idx], dtype=np.float32)
    y_pred = predict_sci_coefficients_group_mlp(
        art,
        coef_near_phys=filtered_triplet["coef_near"][test_idx],
        coef_far_phys=filtered_triplet["coef_far"][test_idx],
        ctx_near_phys=filtered_triplet["ctx_near"][test_idx],
        ctx_far_phys=filtered_triplet["ctx_far"][test_idx],
        ctx_sci_phys=filtered_triplet["ctx_sci"][test_idx],
    )

    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    rows.append(
        {
            "name": cfg["name"],
            "n_epochs": int(cfg["n_epochs"]),
            "patience": int(cfg["patience"]),
            "best_epoch": int(art["best_epoch"]),
            "best_val_loss": float(art["best_val_loss"]),
            "mean_rmse": float(np.nanmean(rmse)),
            "mean_mae": float(np.nanmean(mae)),
        }
    )

sweep_df = pd.DataFrame(rows).sort_values(["best_val_loss", "mean_rmse"], ascending=True).reset_index(drop=True)
print("\nSchedule sweep summary (sorted):")
print(sweep_df.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

In [ ]:
# Auto-suggest deep-group schedule from dataset size and target optimization steps
import math
import pandas as pd

required = ["filtered_triplet"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run filtering/data cells first. Missing: " + ", ".join(missing))

# Inputs you can tune quickly.
BATCH_SIZE = 256
TARGET_TOTAL_STEPS = 360
PATIENCE_RATIO = 0.20
PATIENCE_MIN = 5
PATIENCE_MAX = 12

n_rows = int(np.asarray(filtered_triplet["coef_sci"]).shape[0])
steps_per_epoch = int(math.ceil(n_rows / float(BATCH_SIZE)))

if steps_per_epoch <= 0:
    raise RuntimeError("Invalid steps_per_epoch; check dataset size and batch size")

base_epochs = int(max(20, round(TARGET_TOTAL_STEPS / steps_per_epoch)))
base_patience = int(np.clip(round(PATIENCE_RATIO * base_epochs), PATIENCE_MIN, PATIENCE_MAX))

# Nearby candidates for a lightweight sweep.
candidates = []
for ep in sorted(set([max(20, int(round(base_epochs * 0.75))), base_epochs, int(round(base_epochs * 1.25))])):
    pat = int(np.clip(round(PATIENCE_RATIO * ep), PATIENCE_MIN, PATIENCE_MAX))
    candidates.append({"n_epochs": ep, "patience": pat})

cand_df = pd.DataFrame(candidates)
cand_df["steps_per_epoch"] = steps_per_epoch
cand_df["total_steps"] = cand_df["n_epochs"] * cand_df["steps_per_epoch"]

print("Dataset-size-aware deep-group schedule suggestion")
print(f"  n_rows          = {n_rows}")
print(f"  batch_size      = {BATCH_SIZE}")
print(f"  steps/epoch     = {steps_per_epoch}")
print(f"  baseline epochs = {base_epochs}")
print(f"  baseline patience = {base_patience}")
print("\nCandidate grid for next sweep:")
print(cand_df.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
# Extended optimization sweep for deep group-head MLP (base + residual-corrected)
# Set RUN_EXTENDED_SWEEP=True only when you want to re-run the expensive search.
RUN_EXTENDED_SWEEP = False

if not RUN_EXTENDED_SWEEP:
    if "deep_group_config" in globals() and isinstance(deep_group_config, dict):
        optimized_deep_group_config = dict(deep_group_config)
    else:
        optimized_deep_group_config = {
            "name": "deep_group_mlp_optimized",
            "n_epochs": 30,
            "batch_size": 256,
            "lr": 6e-4,
            "trunk_dims": (768, 384, 192),
            "head_dim": 256,
            "weight_decay": 1e-4,
            "patience": 8,
        }

    if "coef_residual_corrector_alpha" in globals():
        optimized_deep_group_config["residual_alpha"] = float(coef_residual_corrector_alpha)

    print("Extended sweep skipped (RUN_EXTENDED_SWEEP=False).")
    print("Using fallback optimized_deep_group_config:")
    print(optimized_deep_group_config)

else:
    import itertools
    import numpy as np
    import pandas as pd
    from sklearn.linear_model import Ridge

    required = [
        "filtered_triplet",
        "train_coeff_prediction_group_mlp",
        "predict_sci_coefficients_group_mlp",
    ]
    missing = [k for k in required if k not in globals()]
    if missing:
        raise RuntimeError("Run prerequisite model cells first. Missing: " + ", ".join(missing))

    coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
    coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
    coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float32)
    ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
    ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
    ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)

    def _metric_row(y_true, y_pred):
        rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
        mae = np.mean(np.abs(y_pred - y_true), axis=0)
        return float(np.nanmean(rmse)), float(np.nanmean(mae))

    def _build_residual_features(idx, base_pred):
        x_near = coef_near_all[idx]
        x_far = coef_far_all[idx]
        x_sci = ctx_sci_all[idx]
        x_dctx = x_sci - 0.5 * (ctx_near_all[idx] + ctx_far_all[idx])
        return np.hstack([base_pred, x_near, x_far, x_dctx, x_sci]).astype(np.float32)

    # Compact but meaningful grid.
    param_grid = {
        "n_epochs": [20, 25, 30],
        "patience": [5, 8],
        "lr": [6e-4, 1e-3],
        "trunk_dims": [(1024, 512, 256), (768, 384, 192)],
        "head_dim": [192, 256],
        "weight_decay": [1e-4, 3e-4],
    }

    all_cfgs = [
        {
            "n_epochs": ep,
            "patience": pat,
            "lr": lr,
            "trunk_dims": td,
            "head_dim": hd,
            "weight_decay": wd,
        }
        for ep, pat, lr, td, hd, wd in itertools.product(
            param_grid["n_epochs"],
            param_grid["patience"],
            param_grid["lr"],
            param_grid["trunk_dims"],
            param_grid["head_dim"],
            param_grid["weight_decay"],
        )
    ]

    # Keep runtime manageable: deterministic subsample of config space.
    rng = np.random.default_rng(42)
    max_cfg = 14
    if len(all_cfgs) > max_cfg:
        pick = np.sort(rng.choice(len(all_cfgs), size=max_cfg, replace=False))
        cfgs = [all_cfgs[i] for i in pick]
    else:
        cfgs = all_cfgs

    print(f"Running extended optimization sweep on {len(cfgs)} configs")

    rows = []
    for i, cfg in enumerate(cfgs, start=1):
        print(f"\n[{i:02d}/{len(cfgs)}] cfg={cfg}")
        art = train_coeff_prediction_group_mlp(
            filtered_triplet,
            n_epochs=int(cfg["n_epochs"]),
            batch_size=256,
            lr=float(cfg["lr"]),
            trunk_dims=tuple(int(v) for v in cfg["trunk_dims"]),
            head_dim=int(cfg["head_dim"]),
            weight_decay=float(cfg["weight_decay"]),
            grad_clip=1.0,
            patience=int(cfg["patience"]),
            seed=42,
        )

        train_idx = np.asarray(art["train_idx"], dtype=int)
        val_idx = np.asarray(art["val_idx"], dtype=int)
        test_idx = np.asarray(art["test_idx"], dtype=int)

        y_tr = coef_sci_all[train_idx]
        y_va = coef_sci_all[val_idx]
        y_te = coef_sci_all[test_idx]

        base_tr = predict_sci_coefficients_group_mlp(
            art,
            coef_near_phys=coef_near_all[train_idx],
            coef_far_phys=coef_far_all[train_idx],
            ctx_near_phys=ctx_near_all[train_idx],
            ctx_far_phys=ctx_far_all[train_idx],
            ctx_sci_phys=ctx_sci_all[train_idx],
        ).astype(np.float32)
        base_va = predict_sci_coefficients_group_mlp(
            art,
            coef_near_phys=coef_near_all[val_idx],
            coef_far_phys=coef_far_all[val_idx],
            ctx_near_phys=ctx_near_all[val_idx],
            ctx_far_phys=ctx_far_all[val_idx],
            ctx_sci_phys=ctx_sci_all[val_idx],
        ).astype(np.float32)
        base_te = predict_sci_coefficients_group_mlp(
            art,
            coef_near_phys=coef_near_all[test_idx],
            coef_far_phys=coef_far_all[test_idx],
            ctx_near_phys=ctx_near_all[test_idx],
            ctx_far_phys=ctx_far_all[test_idx],
            ctx_sci_phys=ctx_sci_all[test_idx],
        ).astype(np.float32)

        base_rmse, base_mae = _metric_row(y_te, base_te)

        X_tr = _build_residual_features(train_idx, base_tr)
        X_va = _build_residual_features(val_idx, base_va)
        X_te = _build_residual_features(test_idx, base_te)
        res_tr = (y_tr - base_tr).astype(np.float32)

        best = {"alpha": None, "score": np.inf, "model": None}
        for alpha in [0.05, 0.1, 0.3, 1.0, 3.0, 10.0]:
            reg = Ridge(alpha=float(alpha), fit_intercept=True, random_state=42)
            reg.fit(X_tr, res_tr)
            pred_va = np.clip(base_va + reg.predict(X_va).astype(np.float32), 0.0, None)
            score = float(np.mean((pred_va - y_va) ** 2))
            if score < best["score"]:
                best = {"alpha": float(alpha), "score": score, "model": reg}

        pred_te_corr = np.clip(base_te + best["model"].predict(X_te).astype(np.float32), 0.0, None)
        corr_rmse, corr_mae = _metric_row(y_te, pred_te_corr)

        rows.append(
            {
                "n_epochs": int(cfg["n_epochs"]),
                "patience": int(cfg["patience"]),
                "lr": float(cfg["lr"]),
                "trunk_dims": str(tuple(int(v) for v in cfg["trunk_dims"])),
                "head_dim": int(cfg["head_dim"]),
                "weight_decay": float(cfg["weight_decay"]),
                "best_epoch": int(art["best_epoch"]),
                "best_val_loss": float(art["best_val_loss"]),
                "base_mean_rmse": float(base_rmse),
                "base_mean_mae": float(base_mae),
                "corr_mean_rmse": float(corr_rmse),
                "corr_mean_mae": float(corr_mae),
                "residual_alpha": float(best["alpha"]),
            }
        )

    opt_df = pd.DataFrame(rows).sort_values(["corr_mean_rmse", "corr_mean_mae", "base_mean_rmse"], ascending=True).reset_index(drop=True)
    print("\nTop optimization results (sorted):")
    print(opt_df.head(10).to_string(index=False, float_format=lambda v: f"{v:.6g}"))

    best_cfg_row = opt_df.iloc[0]
    optimized_deep_group_config = {
        "name": "deep_group_mlp_optimized",
        "n_epochs": int(best_cfg_row["n_epochs"]),
        "batch_size": 256,
        "lr": float(best_cfg_row["lr"]),
        "trunk_dims": tuple(int(v.strip()) for v in str(best_cfg_row["trunk_dims"]).strip("()").split(",") if v.strip()),
        "head_dim": int(best_cfg_row["head_dim"]),
        "weight_decay": float(best_cfg_row["weight_decay"]),
        "patience": int(best_cfg_row["patience"]),
        "residual_alpha": float(best_cfg_row["residual_alpha"]),
    }

    print("\nRecommended optimized config:")
    print(optimized_deep_group_config)
